# CORA 50M — Entrenamiento en T4
Pipeline completo: Encoder(Mamba,8L) → Crystallizer → CRE(10iter) → Decoder(8L)
~37M parámetros | vocab_size=8000 | 5000 steps | RX6600/T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cora_50m'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'Drive montado. Checkpoints en: {DRIVE_DIR}/checkpoints')

In [ ]:
import os, pathlib, base64, sys

ROOT = pathlib.Path('/content/aion_c')
for pkg in ['core', 'synth', 'encoder', 'crystallizer', 'cre', 'decoder', 'router', 'experiments']:
    (ROOT / pkg).mkdir(parents=True, exist_ok=True)

files_b64 = {
    'core/__init__.py': (
        'IiIiCkFJT04tQyBjb3JlIOKAlCB0aXBvcyB5IGRhdGFjbGFzc2VzIGNvbXBhcnRpZG9zIHBvciB0'
        'b2RvcyBsb3MgbcOzZHVsb3MgZGUgQ09SQS4KIiIiCgpmcm9tIC5ncmFwaCBpbXBvcnQgKAogICAg'
        'Q2F1c2FsRWRnZSwKICAgIENhdXNhbEdyYXBoLAogICAgQ2F1c2FsTm9kZSwKICAgIENhdXNhbFJl'
        'bGF0aW9uLAogICAgTm9kZVR5cGUsCiAgICBDQVVTQUxfUkVMQVRJT05TLAogICAgQ0FVU0FMX1JF'
        'TEFUSU9OU19MSVNULAogICAgQ09OVFJBRElDVElPTl9QQUlSUywKICAgIElOSElCSVRPUllfUkVM'
        'QVRJT05TLAogICAgTk9ERV9UWVBFUywKICAgIFBPU0lUSVZFX1JFTEFUSU9OUywKICAgIFNUUlVD'
        'VFVSQUxfUkVMQVRJT05TLAogICAgU1lNTUVUUklDX1JFTEFUSU9OUywKICAgIFRFTVBPUkFMX1JF'
        'TEFUSU9OUywKKQoKX19hbGxfXyA9IFsKICAgICJDYXVzYWxFZGdlIiwKICAgICJDYXVzYWxHcmFw'
        'aCIsCiAgICAiQ2F1c2FsTm9kZSIsCiAgICAiQ2F1c2FsUmVsYXRpb24iLAogICAgIk5vZGVUeXBl'
        'IiwKICAgICJDQVVTQUxfUkVMQVRJT05TIiwKICAgICJDQVVTQUxfUkVMQVRJT05TX0xJU1QiLAog'
        'ICAgIkNPTlRSQURJQ1RJT05fUEFJUlMiLAogICAgIklOSElCSVRPUllfUkVMQVRJT05TIiwKICAg'
        'ICJOT0RFX1RZUEVTIiwKICAgICJQT1NJVElWRV9SRUxBVElPTlMiLAogICAgIlNUUlVDVFVSQUxf'
        'UkVMQVRJT05TIiwKICAgICJTWU1NRVRSSUNfUkVMQVRJT05TIiwKICAgICJURU1QT1JBTF9SRUxB'
        'VElPTlMiLApdCg=='
    ),
    'core/graph.py': (
        '...'
        'IGl0ZXIoc2VsZi5lZGdlcykK'
    ),
    'synth/__init__.py': (
        'IiIiCkFJT04tQyBzeW50aCDigJQgR2VuZXJhZG9yZXMgZGUgZGF0b3Mgc2ludMOpdGljb3MgcGFy'
        'YSBlbnRyZW5hbWllbnRvLgoiIiIKCmZyb20gLmNhdXNhbF9ncmFwaF9nZW4gaW1wb3J0ICgKICAg'
        'IEFuc3dlclR5cGUsCiAgICBDYXVzYWxFeGFtcGxlLAogICAgQ2F1c2FsR3JhcGhHZW5lcmF0b3Is'
        'CiAgICBWZXJpZmljYXRpb25SZXN1bHQsCiAgICB2ZXJpZnlfZXhhbXBsZSwKKQoKX19hbGxfXyA9'
        'IFsKICAgICJBbnN3ZXJUeXBlIiwKICAgICJDYXVzYWxFeGFtcGxlIiwKICAgICJDYXVzYWxHcmFw'
        'aEdlbmVyYXRvciIsCiAgICAiVmVyaWZpY2F0aW9uUmVzdWx0IiwKICAgICJ2ZXJpZnlfZXhhbXBs'
        'ZSIsCl0K'
    ),
    'synth/causal_graph_gen.py': (
        '...'
        'dmFsdWUpCg=='
    ),
    'encoder/__init__.py': (
        'IiIiCkFJT04tQyBlbmNvZGVyIOKAlCBTdHJlYW1FbmNvZGVyIGJhc2FkbyBlbiBNYW1iYS1zdHls'
        'ZSBTU00uCgpDb252aWVydGUgdG9rZW4gSURzIGVuIGNvbmNlcHQgdmVjdG9ycyBjb24gY29tcGxl'
        'amlkYWQgTyhMKSwKZXZpdGFuZG8gbGEgTyhMwrIpIGRlIGF0dGVudGlvbi4gUGFzbyBwcmV2aW8g'
        'YWwgR3JhcGhDb25zdHJ1Y3Rvci4KIiIiCgpmcm9tIC5tYW1iYV9sYXllciBpbXBvcnQgKAogICAg'
        'R2F0ZWRGRk4sCiAgICBNYW1iYUxheWVyLAogICAgUk1TTm9ybSwKICAgIFNlbGVjdGl2ZVNTTSwK'
        'ICAgIFN0cmVhbUVuY29kZXJDb25maWcsCikKZnJvbSAubW9kZWwgaW1wb3J0IFN0cmVhbUVuY29k'
        'ZXIKCl9fYWxsX18gPSBbCiAgICAiR2F0ZWRGRk4iLAogICAgIk1hbWJhTGF5ZXIiLAogICAgIlJN'
        'U05vcm0iLAogICAgIlNlbGVjdGl2ZVNTTSIsCiAgICAiU3RyZWFtRW5jb2RlciIsCiAgICAiU3Ry'
        'ZWFtRW5jb2RlckNvbmZpZyIsCl0K'
    ),
    'encoder/mamba_layer.py': (
        '...'
        'cm4geCwgc3RhdGVzCg=='
    ),
    'encoder/model.py': (
        'IiIiCmVuY29kZXIvbW9kZWwucHkg4oCUIFN0cmVhbUVuY29kZXI6IHRva2VucyDihpIgY29uY2Vw'
        'dCB2ZWN0b3JzCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09CgpFbCBTdHJlYW1FbmNvZGVyIGVzIGVsIHByaW1lciBtw7NkdWxvIGRlbCBw'
        'aXBlbGluZSBBSU9OLUMgQ0VOOgoKICB0b2tlbl9pZHMgW0IsIExdCiAgICAgICDihpMKICB0b2tl'
        'bl9lbWJlZGRpbmcgW0IsIEwsIERdCiAgICAgICDihpMKICBNYW1iYUxheWVyIMOXIG5fbGF5ZXJz'
        'CiAgICAgICDihpMKICBSTVNOb3JtCiAgICAgICDihpMKICBjb25jZXB0X3Byb2plY3RvciBbQiwg'
        'TCwgY29uY2VwdF9kaW1dCiAgICAgICDihpMKICBjb25jZXB0X3ZlY3RvcnMg4oaSIEdyYXBoQ29u'
        'c3RydWN0b3IKClByb3BpZWRhZCBjbGF2ZTogbWVtb3JpYSBPKEwpLCBubyBPKEzCsikuCkVsIHNj'
        'YW4gc2VjdWVuY2lhbCBkZWwgU1NNIG1hbnRpZW5lIHVuIGVzdGFkbyBoW0IsIERfaW5uZXIsIE5d'
        'Cih0YW1hw7FvIGNvbnN0YW50ZSkgbWllbnRyYXMgcHJvY2VzYSBjYWRhIHRva2VuLiBMYSBpbmZv'
        'cm1hY2nDs24KZGVsIHBhc2FkbyBxdWVkYSBjb21wcmltaWRhIGVuIGVzZSB2ZWN0b3IgZGUgZXN0'
        'YWRvLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5n'
        'IGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmltcG9ydCB0b3JjaAppbXBvcnQg'
        'dG9yY2gubm4gYXMgbm4KCmZyb20gdG9yY2gudXRpbHMuY2hlY2twb2ludCBpbXBvcnQgY2hlY2tw'
        'b2ludCBhcyBfY2twdAoKZnJvbSAubWFtYmFfbGF5ZXIgaW1wb3J0IEdhdGVkRkZOLCBNYW1iYUxh'
        'eWVyLCBSTVNOb3JtLCBTdHJlYW1FbmNvZGVyQ29uZmlnCgoKIyDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBTVFJFQU0gRU5DT0RF'
        'UgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgAoKY2xhc3MgU3RyZWFtRW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBDb252'
        'aWVydGUgdG9rZW4gSURzIGVuIGNvbmNlcHQgdmVjdG9ycyB1c2FuZG8gTWFtYmEtc3R5bGUgU1NN'
        'LgoKICAgIEVsIGNvbmNlcHRvIGRlICJlc3BhY2lvIGNvbmNlcHR1YWwiIChjb25jZXB0X2RpbSA8'
        'IGhpZGRlbl9kaW0pOgogICAgICAidGhlIHF1aWNrIGJyb3duIGZveCIgPSA0IHRva2VucyDihpIg'
        'Mi0zIGNvbmNlcHRvcy4KICAgICAgTGEgY29tcHJlc2nDs24gZWxpbWluYSBpbmZvcm1hY2nDs24g'
        'bMOpeGljYSBzdXBlcmZpY2lhbCwKICAgICAgZGVqYW5kbyBzb2xvIGxhIHNlbcOhbnRpY2EgcmVs'
        'ZXZhbnRlIHBhcmEgZWwgQ0VDLgogICAgICBFbCBDRUMgb3BlcmEgZW4gY29uY2VwdF9kaW09MTI4'
        'ZCwgbm8gZW4gaGlkZGVuX2RpbT0yNTZkLgogICAgICDihpIgNHggbWVub3MgY29tcHV0ZSBwb3Ig'
        'b3BlcmFjacOzbiBlbiBlbCByYXpvbmFkb3IuCgogICAgQ29uZmlndXJhY2nDs24gdGlueSAodGVz'
        'dGluZyk6CiAgICAgIHZvY2FiX3NpemU9MzIwMDAsIGhpZGRlbl9kaW09MjU2LCBuX2xheWVycz00'
        'LAogICAgICBzdGF0ZV9kaW09MTYsIGNvbmNlcHRfZGltPTEyOAogICAgICBQYXLDoW1ldHJvcyB0'
        'b3RhbGVzOiB+MTNNCgogICAgVXNvOgogICAgICAgIGNvbmZpZyAgPSBTdHJlYW1FbmNvZGVyQ29u'
        'ZmlnKCkKICAgICAgICBlbmNvZGVyID0gU3RyZWFtRW5jb2Rlcihjb25maWcpCiAgICAgICAgaWRz'
        'ICAgICA9IHRvcmNoLnJhbmRpbnQoMCwgY29uZmlnLnZvY2FiX3NpemUsICgyLCA1MTIpKQogICAg'
        'ICAgIHZlY3MgICAgPSBlbmNvZGVyKGlkcykgICAjIFsyLCA1MTIsIDEyOF0KICAgICIiIgoKICAg'
        'IGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWc6IFN0cmVhbUVuY29kZXJDb25maWcpIC0+IE5vbmU6'
        'CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcK'
        'CiAgICAgICAgIyBFbWJlZGRpbmcgZGUgdG9rZW5zCiAgICAgICAgc2VsZi50b2tlbl9lbWJlZGRp'
        'bmcgPSBubi5FbWJlZGRpbmcoY29uZmlnLnZvY2FiX3NpemUsIGNvbmZpZy5oaWRkZW5fZGltKQoK'
        'ICAgICAgICAjIFN0YWNrIGRlIGNhcGFzIE1hbWJhCiAgICAgICAgc2VsZi5sYXllcnMgPSBubi5N'
        'b2R1bGVMaXN0KFsKICAgICAgICAgICAgTWFtYmFMYXllcihjb25maWcpIGZvciBfIGluIHJhbmdl'
        'KGNvbmZpZy5uX2xheWVycykKICAgICAgICBdKQoKICAgICAgICAjIE5vcm1hbGl6YWNpw7NuIGZp'
        'bmFsCiAgICAgICAgc2VsZi5ub3JtID0gUk1TTm9ybShjb25maWcuaGlkZGVuX2RpbSwgZXBzPWNv'
        'bmZpZy5ybXNfZXBzKQoKICAgICAgICAjIFByb3llY2Npw7NuIGFsIGVzcGFjaW8gY29uY2VwdHVh'
        'bCAoaGlkZGVuX2RpbSDihpIgY29uY2VwdF9kaW0pCiAgICAgICAgIyBTaW4gYmlhcyDigJQgbGEg'
        'aW5mb3JtYWNpw7NuIHZpZW5lIGRlbCBlbWJlZGRpbmcgKyBTU00KICAgICAgICBzZWxmLmNvbmNl'
        'cHRfcHJvamVjdG9yID0gbm4uTGluZWFyKAogICAgICAgICAgICBjb25maWcuaGlkZGVuX2RpbSwg'
        'Y29uZmlnLmNvbmNlcHRfZGltLCBiaWFzPUZhbHNlCiAgICAgICAgKQoKICAgICAgICBzZWxmLmdy'
        'YWRpZW50X2NoZWNrcG9pbnRpbmcgPSBGYWxzZQogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygp'
        'CgogICAgIyDilIDilIAgSW5pY2lhbGl6YWNpw7NuIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBlbmFibGVfZ3JhZGllbnRfY2hlY2tw'
        'b2ludGluZyhzZWxmKSAtPiBOb25lOgogICAgICAgICIiIgogICAgICAgIEFjdGl2YSBncmFkaWVu'
        'dCBjaGVja3BvaW50aW5nIGVuIGxhcyBjYXBhcyBNYW1iYUxheWVyLgogICAgICAgIFJlY29tcHV0'
        'YSBsYXMgYWN0aXZhY2lvbmVzIGR1cmFudGUgZWwgYmFja3dhcmQgZW4gbHVnYXIgZGUgZ3VhcmRh'
        'cmxhcywKICAgICAgICByZWR1Y2llbmRvIGVsIHVzbyBkZSBWUkFNIGRlbCBiYWNrd2FyZCB+Mi0z'
        'w5cgYSBjb3N0YSBkZSB+MTUlIG3DoXMgY29tcHV0ZS4KICAgICAgICBTb2xvIGFjdGl2byBlbiBt'
        'b2RvIHRyYWluaW5nLgogICAgICAgICIiIgogICAgICAgIHNlbGYuZ3JhZGllbnRfY2hlY2twb2lu'
        'dGluZyA9IFRydWUKCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhzZWxmKSAtPiBOb25lOgogICAgICAg'
        'ICIiIgogICAgICAgIEluaWNpYWxpemFjacOzbiBlc3TDoW5kYXIgcGFyYSBMTE1zOgogICAgICAg'
        'IC0gRW1iZWRkaW5nOiBub3JtYWwoc3RkPTAuMDIpCiAgICAgICAgLSBjb25jZXB0X3Byb2plY3Rv'
        'cjogbm9ybWFsKHN0ZD0wLjAyIC8gc3FydCgyICogbl9sYXllcnMpKQogICAgICAgICAgRWwgZmFj'
        'dG9yIDEvc3FydCgyTCkgcmVkdWNlIGxhIHZhcmlhbnphIGRlIGxvcyByZXNpZHVhbHMKICAgICAg'
        'ICAgIGFsIGFjdW11bGFyIEwgY2FwYXMgKHNpbWlsYXIgYSBHUFQtMikuCiAgICAgICAgIiIiCiAg'
        'ICAgICAgc3RkX2VtYiAgPSAwLjAyCiAgICAgICAgc3RkX3Byb2ogPSAwLjAyIC8gKDIgKiBzZWxm'
        'LmNvbmZpZy5uX2xheWVycykgKiogMC41CgogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLnRv'
        'a2VuX2VtYmVkZGluZy53ZWlnaHQsIHN0ZD1zdGRfZW1iKQogICAgICAgIG5uLmluaXQubm9ybWFs'
        'XyhzZWxmLmNvbmNlcHRfcHJvamVjdG9yLndlaWdodCwgc3RkPXN0ZF9wcm9qKQoKICAgICMg4pSA'
        '4pSAIEZvcndhcmQgcHJpbmNpcGFsIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgAoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHRva2VuX2lkczogdG9yY2guVGVuc29yKSAt'
        'PiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgdG9r'
        'ZW5faWRzOiBbQiwgTF0g4oCUIGxvbmcgdGVuc29yIGRlIElEcyBkZSB0b2tlbnMKCiAgICAgICAg'
        'UmV0dXJuczoKICAgICAgICAgICAgY29uY2VwdF92ZWN0b3JzOiBbQiwgTCwgY29uY2VwdF9kaW1d'
        'CiAgICAgICAgICAgICAgQ2FkYSBwb3NpY2nDs24gbCBjb250aWVuZSBlbCBjb25jZXB0byBleHRy'
        'YcOtZG8gZGVsIGNvbnRleHRvCiAgICAgICAgICAgICAgWzAuLmxdIChncmFjaWFzIGFsIHNjYW4g'
        'Y2F1c2FsIGRlbCBTU00pLgogICAgICAgICIiIgogICAgICAgIHggPSBzZWxmLnRva2VuX2VtYmVk'
        'ZGluZyh0b2tlbl9pZHMpICAjIFtCLCBMLCBEXQoKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5s'
        'YXllcnM6CiAgICAgICAgICAgIGlmIHNlbGYuZ3JhZGllbnRfY2hlY2twb2ludGluZyBhbmQgc2Vs'
        'Zi50cmFpbmluZzoKICAgICAgICAgICAgICAgIHgsIF8gPSBfY2twdChsYXllciwgeCwgdXNlX3Jl'
        'ZW50cmFudD1GYWxzZSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHgsIF8gPSBs'
        'YXllcih4KQoKICAgICAgICB4ID0gc2VsZi5ub3JtKHgpCiAgICAgICAgcmV0dXJuIHNlbGYuY29u'
        'Y2VwdF9wcm9qZWN0b3IoeCkgICAgICMgW0IsIEwsIGNvbmNlcHRfZGltXQoKICAgICMg4pSA4pSA'
        'IEZvcndhcmQgY29uIGluc3BlY2Npw7NuIGludGVybmEg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIGZvcndhcmRfd2l0'
        'aF9zdGF0ZXMoCiAgICAgICAgc2VsZiwKICAgICAgICB0b2tlbl9pZHM6IHRvcmNoLlRlbnNvciwK'
        'ICAgICkgLT4gVHVwbGVbdG9yY2guVGVuc29yLCBMaXN0W3RvcmNoLlRlbnNvcl1dOgogICAgICAg'
        'ICIiIgogICAgICAgIElndWFsIHF1ZSBmb3J3YXJkKCkgcGVybyByZXRvcm5hIHRhbWJpw6luIGxh'
        'IHRyYXllY3RvcmlhIGRlIGVzdGFkb3MgU1NNLgoKICAgICAgICBVc286CiAgICAgICAgICAgIGNv'
        'bmNlcHRzLCBzdGF0ZXMgPSBlbmNvZGVyLmZvcndhcmRfd2l0aF9zdGF0ZXMoaWRzKQogICAgICAg'
        'ICAgICAjIHN0YXRlc1tpXTogW0IsIEwsIERfaW5uZXIsIE5dIOKAlCBlc3RhZG9zIGRlIGxhIGNh'
        'cGEgaQogICAgICAgICAgICAjIHN0YXRlc1tpXVs6LCB0LCA6LCA6XSDigJQgZXN0YWRvIGVuIGVs'
        'IHRpbWVzdGVwIHQsIGNhcGEgaQoKICAgICAgICBVdGlsIHBhcmE6CiAgICAgICAgICAgIC0gVmVy'
        'aWZpY2FyIHF1ZSBlbCBlc3RhZG8gU1NNIGNhbWJpYSBlbnRyZSB0aW1lc3RlcHMKICAgICAgICAg'
        'ICAgLSBEZWJ1Z2dpbmcgZGVsIHNjYW4KICAgICAgICAgICAgLSBWaXN1YWxpemFjacOzbiBkZSBx'
        'dcOpIGluZm9ybWFjacOzbiByZXRpZW5lIGVsIGVzdGFkbwogICAgICAgICIiIgogICAgICAgIHgg'
        'PSBzZWxmLnRva2VuX2VtYmVkZGluZyh0b2tlbl9pZHMpCiAgICAgICAgYWxsX3N0YXRlczogTGlz'
        'dFt0b3JjaC5UZW5zb3JdID0gW10KCiAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzOgog'
        'ICAgICAgICAgICB4LCBzdGF0ZXMgPSBsYXllcih4LCByZXR1cm5fc3RhdGVzPVRydWUpCiAgICAg'
        'ICAgICAgIGFsbF9zdGF0ZXMuYXBwZW5kKHN0YXRlcykgICAjIHN0YXRlczogW0IsIEwsIERfaW5u'
        'ZXIsIE5dCgogICAgICAgIHggPSBzZWxmLm5vcm0oeCkKICAgICAgICBjb25jZXB0cyA9IHNlbGYu'
        'Y29uY2VwdF9wcm9qZWN0b3IoeCkKICAgICAgICByZXR1cm4gY29uY2VwdHMsIGFsbF9zdGF0ZXMK'
        'CiAgICBkZWYgZ2V0X3NzbV9BX2Jhcl9udW1lbChzZWxmLCB0b2tlbl9pZHM6IHRvcmNoLlRlbnNv'
        'cikgLT4gaW50OgogICAgICAgICIiIgogICAgICAgIEVqZWN1dGEgdW4gZm9yd2FyZCB5IHJldG9y'
        'bmEgZWwgbsO6bWVybyBkZSBlbGVtZW50b3MgZGVsIHRlbnNvciBBX2JhcgogICAgICAgIGNvbXB1'
        'dGFkbyBlbiBsYSBwcmltZXJhIGNhcGEgU1NNLgoKICAgICAgICBVc2FkbyBlbiB0ZXN0cyBkZSBl'
        'c2NhbGFkbyBkZSBtZW1vcmlhOgogICAgICAgICAgQV9iYXIubnVtZWwoKSA9IEIgw5cgTCDDlyBE'
        'X2lubmVyIMOXIE4gIOKGkCBsaW5lYWwgZW4gTCwgTk8gY3VhZHLDoXRpY28KCiAgICAgICAgRGV2'
        'dWVsdmUgc2llbXByZSBlbCB2YWxvciBkZSBsYSBQUklNRVJBIGNhcGEgKGxheWVyc1swXS5zc20p'
        'LgogICAgICAgICIiIgogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBz'
        'ZWxmLmZvcndhcmQodG9rZW5faWRzKQogICAgICAgIHJldHVybiBzZWxmLmxheWVyc1swXS5zc20u'
        'X2xhc3RfQV9iYXJfbnVtZWwKCiAgICAjIOKUgOKUgCBVdGlsaWRhZGVzIOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAg'
        'IGRlZiBjb3VudF9wYXJhbWV0ZXJzKHNlbGYpIC0+IGludDoKICAgICAgICAiIiJOw7ptZXJvIHRv'
        'dGFsIGRlIHBhcsOhbWV0cm9zIGVudHJlbmFibGVzLiIiIgogICAgICAgIHJldHVybiBzdW0ocC5u'
        'dW1lbCgpIGZvciBwIGluIHNlbGYucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKCiAg'
        'ICBkZWYgcGFyYW1ldGVyX2JyZWFrZG93bihzZWxmKSAtPiBEaWN0W3N0ciwgaW50XToKICAgICAg'
        'ICAiIiJEZXNnbG9zZSBkZSBwYXLDoW1ldHJvcyBwb3IgbcOzZHVsby4iIiIKICAgICAgICByZXR1'
        'cm4gewogICAgICAgICAgICAidG9rZW5fZW1iZWRkaW5nIjogc2VsZi50b2tlbl9lbWJlZGRpbmcu'
        'd2VpZ2h0Lm51bWVsKCksCiAgICAgICAgICAgICJsYXllcnNfc3NtIjogICAgICBzdW0oCiAgICAg'
        'ICAgICAgICAgICBwLm51bWVsKCkKICAgICAgICAgICAgICAgIGZvciBsYXllciBpbiBzZWxmLmxh'
        'eWVycwogICAgICAgICAgICAgICAgZm9yIHAgaW4gbGF5ZXIuc3NtLnBhcmFtZXRlcnMoKQogICAg'
        'ICAgICAgICApLAogICAgICAgICAgICAibGF5ZXJzX2ZmbiI6ICAgICAgc3VtKAogICAgICAgICAg'
        'ICAgICAgcC5udW1lbCgpCiAgICAgICAgICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnMK'
        'ICAgICAgICAgICAgICAgIGZvciBwIGluIGxheWVyLmZmbi5wYXJhbWV0ZXJzKCkKICAgICAgICAg'
        'ICAgKSwKICAgICAgICAgICAgImxheWVyc19ub3JtcyI6ICAgIHN1bSgKICAgICAgICAgICAgICAg'
        'IHAubnVtZWwoKQogICAgICAgICAgICAgICAgZm9yIGxheWVyIGluIHNlbGYubGF5ZXJzCiAgICAg'
        'ICAgICAgICAgICBmb3IgcCBpbiBsaXN0KGxheWVyLm5vcm0xLnBhcmFtZXRlcnMoKSkgKyBsaXN0'
        'KGxheWVyLm5vcm0yLnBhcmFtZXRlcnMoKSkKICAgICAgICAgICAgKSwKICAgICAgICAgICAgImNv'
        'bmNlcHRfcHJvamVjdG9yIjogc2VsZi5jb25jZXB0X3Byb2plY3Rvci53ZWlnaHQubnVtZWwoKSwK'
        'ICAgICAgICAgICAgInRvdGFsIjogc2VsZi5jb3VudF9wYXJhbWV0ZXJzKCksCiAgICAgICAgfQo='
    ),
    'crystallizer/__init__.py': (
        'IiIiCkFJT04tQyBjcnlzdGFsbGl6ZXIg4oCUIEdyYXBoQ3J5c3RhbGxpemVyOiBjb25jZXB0IHZl'
        'Y3RvcnMg4oaSIENhdXNhbEdyYXBoLgoKQ29udmllcnRlIGNvbmNlcHQgdmVjdG9ycyBbQiwgTCwg'
        'RF0gKGRlbCBTdHJlYW1FbmNvZGVyKSBlbiBncmFmb3MgY2F1c2FsZXMKZXN0cnVjdHVyYWRvcywg'
        'dXNhbmRvIGRldGVjY2nDs24gZGUgbm9kb3MsIHBvb2xpbmcgcG9yIGNyb3NzLWF0dGVudGlvbgp5'
        'IHB1bnR1YWNpw7NuIGRlIHJlbGFjaW9uZXMgYXNpbcOpdHJpY2FzLgoiIiIKCmZyb20gLmNvbmZp'
        'ZyBpbXBvcnQgQ3J5c3RhbGxpemVyQ29uZmlnCmZyb20gLm1vZGVsIGltcG9ydCBDcnlzdGFsbGl6'
        'ZXJPdXRwdXQsIEdyYXBoQ3J5c3RhbGxpemVyCmZyb20gLm5vZGVfZGV0ZWN0b3IgaW1wb3J0IE5v'
        'ZGVEZXRlY3Rvcgpmcm9tIC5wb29sZXIgaW1wb3J0IENyb3NzQXR0ZW50aW9uUG9vbGVyCmZyb20g'
        'LnJlbGF0aW9uX3Njb3JlciBpbXBvcnQgQXN5bW1ldHJpY1JlbGF0aW9uU2NvcmVyCgpfX2FsbF9f'
        'ID0gWwogICAgIkFzeW1tZXRyaWNSZWxhdGlvblNjb3JlciIsCiAgICAiQ3Jvc3NBdHRlbnRpb25Q'
        'b29sZXIiLAogICAgIkNyeXN0YWxsaXplckNvbmZpZyIsCiAgICAiQ3J5c3RhbGxpemVyT3V0cHV0'
        'IiwKICAgICJHcmFwaENyeXN0YWxsaXplciIsCiAgICAiTm9kZURldGVjdG9yIiwKXQo='
    ),
    'crystallizer/config.py': (
        'IiIiCmNyeXN0YWxsaXplci9jb25maWcucHkg4oCUIENvbmZpZ3VyYWNpw7NuIGRlbCBHcmFwaENy'
        'eXN0YWxsaXplcgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoK'
        'ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKCkBkYXRhY2xhc3MKY2xh'
        'c3MgQ3J5c3RhbGxpemVyQ29uZmlnOgogICAgIiIiCiAgICBIaXBlcnBhcsOhbWV0cm9zIGRlbCBH'
        'cmFwaENyeXN0YWxsaXplci4KCiAgICBEaXNlw7FhZG8gcGFyYSBjb2luY2lkaXIgY29uIGVsIFN0'
        'cmVhbUVuY29kZXIgdXBzdHJlYW06CiAgICAgICAgaGlkZGVuX2RpbSA9IFN0cmVhbUVuY29kZXJD'
        'b25maWcuY29uY2VwdF9kaW0gPSAxMjggIChvIDI1NiBlbiB0aW55KQogICAgICAgIG5fcmVsYXRp'
        'b25fdHlwZXMgPSBsZW4oQ0FVU0FMX1JFTEFUSU9OUykgPSAxNgogICAgICAgIG5fbm9kZV90eXBl'
        'cyAgICAgPSBsZW4oTm9kZVR5cGUpICAgICAgICAgPSA3CgogICAgQ29uZmlndXJhY2nDs24gdGlu'
        'eSBwYXJhIHRlc3Rpbmc6CiAgICAgICAgaGlkZGVuX2RpbT0yNTYsIG1heF9ub2Rlcz0zMgoKICAg'
        'IFVtYnJhbGVzOgogICAgICAgIG5vZGVfdGhyZXNob2xkICDigJQgc2lnbW9pZCBzY29yZSBtw61u'
        'aW1vIHBhcmEgY29uc2lkZXJhciB1biBub2RvCiAgICAgICAgZWRnZV90aHJlc2hvbGQgIOKAlCBz'
        'aWdtb2lkIHNjb3JlIG3DrW5pbW8gcGFyYSBhZ3JlZ2FyIHVuYSBhcmlzdGEKICAgICAgICBWYWxv'
        'cmVzIGJham9zICgwLjMpIGdhcmFudGl6YW4gZ3JhZm9zIG5vIHZhY8Otb3MgZW4gdGVzdHMgY29u'
        'IHBlc29zIHJhbmRvbS4KICAgICIiIgoKICAgICMgRGltZW5zaW9uZXMKICAgIGhpZGRlbl9kaW06'
        'IGludCAgID0gMjU2ICAgIyA9IGNvbmNlcHRfZGltIGRlbCBTdHJlYW1FbmNvZGVyCiAgICBtYXhf'
        'bm9kZXM6IGludCAgICA9IDMyICAgICMgbcOheGltbyBkZSBub2RvcyBwb3IgZ3JhZm8gb3V0cHV0'
        'CgogICAgIyBWb2NhYnVsYXJpb3MgKGRlYmVuIGNvaW5jaWRpciBjb24gY29yZS9ncmFwaC5weSkK'
        'ICAgIG5fcmVsYXRpb25fdHlwZXM6IGludCA9IDE2ICAgIyBsZW4oQ0FVU0FMX1JFTEFUSU9OUykK'
        'ICAgIG5fbm9kZV90eXBlczogICAgIGludCA9IDcgICAgIyBsZW4oTm9kZVR5cGUpCgogICAgIyBV'
        'bWJyYWxlcyBkZSBkZXRlY2Npw7NuCiAgICBub2RlX3RocmVzaG9sZDogZmxvYXQgPSAwLjMgICMg'
        'cHJvYmFiaWxpZGFkIG3DrW5pbWEgcGFyYSBzZXIgbm9kbwogICAgZWRnZV90aHJlc2hvbGQ6IGZs'
        'b2F0ID0gMC4zICAjIHNpZ21vaWQgc2NvcmUgbcOtbmltbyBwYXJhIHNlciBhcmlzdGEKCiAgICAj'
        'IEFycXVpdGVjdHVyYSBkZWwgcG9vbGVyCiAgICBwb29sZXJfaGVhZHM6IGludCA9IDQgICAgICAg'
        'ICMgY2FiZXphcyBlbiBDcm9zc0F0dGVudGlvblBvb2xlcgoKICAgICMgQXJxdWl0ZWN0dXJhIGRl'
        'bCByZWxhdGlvbiBzY29yZXIKICAgIHJlbGF0aW9uX2hpZGRlbl9kaW06IGludCA9IDY0ICAjIGRp'
        'bWVuc2nDs24gcG9yIHJlbGFjacOzbiBlbiBBc3ltbWV0cmljUmVsYXRpb25TY29yZXIKCiAgICAj'
        'IEFycXVpdGVjdHVyYSBpbnRlcm5hIGRlbCBOb2RlRGV0ZWN0b3IKICAgIG5vZGVfY29uZmlkZW5j'
        'ZV9oaWRkZW5fZGltOiBpbnQgPSA2NAoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5v'
        'bmU6CiAgICAgICAgaWYgc2VsZi5oaWRkZW5fZGltICUgc2VsZi5wb29sZXJfaGVhZHMgIT0gMDoK'
        'ICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiaGlkZGVuX2Rp'
        'bSAoe3NlbGYuaGlkZGVuX2RpbX0pIG11c3QgYmUgZGl2aXNpYmxlIGJ5ICIKICAgICAgICAgICAg'
        'ICAgIGYicG9vbGVyX2hlYWRzICh7c2VsZi5wb29sZXJfaGVhZHN9KSIKICAgICAgICAgICAgKQog'
        'ICAgICAgIGlmIG5vdCAwLjAgPCBzZWxmLm5vZGVfdGhyZXNob2xkIDwgMS4wOgogICAgICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKGYibm9kZV90aHJlc2hvbGQgbXVzdCBiZSBpbiAoMCwxKSwgZ290'
        'IHtzZWxmLm5vZGVfdGhyZXNob2xkfSIpCiAgICAgICAgaWYgbm90IDAuMCA8IHNlbGYuZWRnZV90'
        'aHJlc2hvbGQgPCAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJlZGdlX3RocmVz'
        'aG9sZCBtdXN0IGJlIGluICgwLDEpLCBnb3Qge3NlbGYuZWRnZV90aHJlc2hvbGR9IikK'
    ),
    'crystallizer/node_detector.py': (
        'IiIiCmNyeXN0YWxsaXplci9ub2RlX2RldGVjdG9yLnB5IOKAlCBOb2RlRGV0ZWN0b3IKPT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpNTFAgcXVlIGNsYXNpZmlj'
        'YSBjYWRhIHBvc2ljacOzbiBkZSBsYSBzZWN1ZW5jaWEgY29tbyBub2RvIG8gbm8tbm9kby4KClBv'
        'ciBxdcOpIG5vIHVzYXIgYXR0ZW50aW9uIHBhcmEgZGV0ZWNjacOzbiBkZSBub2RvczoKICAgIExh'
        'IGRldGVjY2nDs24gZGUgbm9kb3MgZXMgdW4gcHJvYmxlbWEgUE9SIFBPU0lDScOTTiDigJQgwr9l'
        'cyBlc3RlIGNvbmNlcHRvCiAgICByZWxldmFudGUgcG9yIHPDrSBtaXNtbz8gTm8gZGVwZW5kZSBk'
        'ZSBzdSByZWxhY2nDs24gY29uIG90cm9zLgogICAgVW4gTUxQIHBvciBwb3NpY2nDs24gZXMgc3Vm'
        'aWNpZW50ZSB5IG3DoXMgZWZpY2llbnRlIHF1ZSBzZWxmLWF0dGVudGlvbi4KClRyZXMgc2FsaWRh'
        'cyBwb3IgcG9zaWNpw7NuOgogICAgbm9kZV9zY29yZSAg4oCUIMK/ZXMgZXN0ZSBjb25jZXB0byB1'
        'biBub2RvIGRlbCBncmFmbz8KICAgIHR5cGVfbG9naXRzIOKAlCDCv3F1w6kgdGlwbyBkZSBub2Rv'
        'IGVzPyAoRU5USVRZLCBFVkVOVCwgU1RBVEUsIC4uLikKICAgIGNvbmZpZGVuY2UgIOKAlCDCv2N1'
        'w6FudG8gY29uZsOtYSBlbCBkZXRlY3RvciBlbiBzdSBwcm9waWEgcHJlZGljY2nDs24/CgpMYSBz'
        'ZXBhcmFjacOzbiBkZSBgbm9kZV9zY29yZWAgeSBgY29uZmlkZW5jZWAgcGVybWl0ZToKICAgIC0g'
        'Tm9kbyBjb24gYWx0YSBwcm9iYWJpbGlkYWQgcGVybyBiYWphIGNvbmZpYW56YSAoaW5jZXJ0aWR1'
        'bWJyZSBnZW51aW5hKQogICAgLSBOb2RvIGNvbiBiYWphIHByb2JhYmlsaWRhZCBwZXJvIGFsdGEg'
        'Y29uZmlhbnphIChkZXRlY3RhIHF1ZSBOTyBlcyBub2RvKQoiIiIKCmZyb20gX19mdXR1cmVfXyBp'
        'bXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5nIGltcG9ydCBUdXBsZQoKaW1wb3J0IHRvcmNo'
        'CmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFsbGl6ZXJD'
        'b25maWcKCgpjbGFzcyBOb2RlRGV0ZWN0b3Iobm4uTW9kdWxlKToKICAgICIiIgogICAgTUxQIHF1'
        'ZSBhc2lnbmEgdW4gcHVudGFqZSBkZSBub2RvIHkgdGlwbyBhIGNhZGEgcG9zaWNpw7NuLgoKICAg'
        'IFVzbzoKICAgICAgICBjZmcgICAgID0gQ3J5c3RhbGxpemVyQ29uZmlnKCkKICAgICAgICBkZXRl'
        'Y3RvciA9IE5vZGVEZXRlY3RvcihjZmcpCiAgICAgICAgY29uY2VwdHMgPSB0b3JjaC5yYW5kbigy'
        'LCA2NCwgMjU2KSAgICAgICAjIFtCLCBMLCBEXQogICAgICAgIHNjb3JlcywgdHlwZV9sb2dpdHMs'
        'IGNvbmYgPSBkZXRlY3Rvcihjb25jZXB0cykKICAgICAgICAjIHNjb3JlczogICAgICBbMiwgNjRd'
        'ICAgICAgIOKAlCBzaWdtb2lkLCBwcm9iYWJpbGlkYWQgZGUgbm9kbwogICAgICAgICMgdHlwZV9s'
        'b2dpdHM6IFsyLCA2NCwgN10gICAg4oCUIHNpbiBhY3RpdmFjacOzbiwgcGFyYSBDcm9zc0VudHJv'
        'cHkKICAgICAgICAjIGNvbmY6ICAgICAgICBbMiwgNjRdICAgICAgIOKAlCBzaWdtb2lkLCBjb25m'
        'aWFuemEgZGVsIGRldGVjdG9yCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmln'
        'OiBDcnlzdGFsbGl6ZXJDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygp'
        'CiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGltCiAgICAgICAgQyA9IGNvbmZpZy5ub2RlX2Nv'
        'bmZpZGVuY2VfaGlkZGVuX2RpbQoKICAgICAgICAjIOKUgOKUgCBQdW50dWFjacOzbiBkZSBub2Rv'
        'IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgRG9zIGNhcGFzOiBs'
        'YSBwcmltZXJhIGV4cGFuZGUgcGFyYSBjYXB0dXJhciBpbnRlcmFjY2lvbmVzIGludGVybmFzLAog'
        'ICAgICAgICMgbGEgc2VndW5kYSBjb2xhcHNhIGEgdW4gZXNjYWxhci4gU2luIHNlc2dvIGVuIGxh'
        'IHNlZ3VuZGEgY2FwYQogICAgICAgICMgcGFyYSBldml0YXIgcXVlIGVsIGJpYXMgZW1wdWplIHRv'
        'ZG9zIGxvcyBzY29yZXMgaGFjaWEgYXJyaWJhLgogICAgICAgIHNlbGYubm9kZV9zY29yZXIgPSBu'
        'bi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRCwgRCksCiAgICAgICAgICAgIG5u'
        'LkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKEQsIDEsIGJpYXM9RmFsc2UpLAogICAgICAg'
        'ICkKCiAgICAgICAgIyDilIDilIAgQ2xhc2lmaWNhZG9yIGRlIHRpcG8g4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSACiAgICAgICAgIyBQcm95ZWNjacOzbiBkaXJlY3RhIEQg4oaSIG5fdHlwZXM7'
        'IG5vIHNpZ21vaWQvc29mdG1heCBhcXXDrQogICAgICAgICMgKGVsIGNvbnN1bWlkb3IgYXBsaWNh'
        'IHNvZnRtYXggbyBhcmdtYXggc2Vnw7puIGVsIHVzbykKICAgICAgICBzZWxmLnR5cGVfY2xhc3Np'
        'ZmllciA9IG5uLkxpbmVhcihELCBjb25maWcubl9ub2RlX3R5cGVzKQoKICAgICAgICAjIOKUgOKU'
        'gCBFc3RpbWFkb3IgZGUgY29uZmlhbnphIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMg'
        'U2VwYXJhZG8gZGVsIHNjb3JlciBwYXJhIGRlc2Fjb3BsYXIgIsK/ZXMgbm9kbz8iIGRlICLCv2N1'
        'w6FudG8gY29uZsOtbz8iCiAgICAgICAgc2VsZi5jb25maWRlbmNlX2hlYWQgPSBubi5TZXF1ZW50'
        'aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoRCwgQyksCiAgICAgICAgICAgIG5uLkdFTFUoKSwK'
        'ICAgICAgICAgICAgbm4uTGluZWFyKEMsIDEsIGJpYXM9RmFsc2UpLAogICAgICAgICkKCiAgICAg'
        'ICAgc2VsZi5faW5pdF93ZWlnaHRzKCkKCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhzZWxmKSAtPiBO'
        'b25lOgogICAgICAgIGZvciBtIGluIHNlbGYubW9kdWxlcygpOgogICAgICAgICAgICBpZiBpc2lu'
        'c3RhbmNlKG0sIG5uLkxpbmVhcik6CiAgICAgICAgICAgICAgICBubi5pbml0Lm5vcm1hbF8obS53'
        'ZWlnaHQsIHN0ZD0wLjAyKQogICAgICAgICAgICAgICAgaWYgbS5iaWFzIGlzIG5vdCBOb25lOgog'
        'ICAgICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG0uYmlhcykKCiAgICBkZWYgZm9yd2Fy'
        'ZCgKICAgICAgICBzZWxmLAogICAgICAgIGNvbmNlcHRzOiB0b3JjaC5UZW5zb3IsCiAgICApIC0+'
        'IFR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgICAg'
        'ICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGNvbmNlcHRzOiBbQiwgTCwgRF0g4oCUIGNv'
        'bmNlcHQgdmVjdG9ycyBkZWwgU3RyZWFtRW5jb2RlcgoKICAgICAgICBSZXR1cm5zOgogICAgICAg'
        'ICAgICBub2RlX3Njb3JlczogIFtCLCBMXSAgICAgICAgICAg4oCUIHNpZ21vaWQg4oiIICgwLCAx'
        'KSwgcHJvYmFiaWxpdHkgb2YgYmVpbmcgYSBub2RlCiAgICAgICAgICAgIHR5cGVfbG9naXRzOiAg'
        'W0IsIEwsIG5fdHlwZXNdICDigJQgc2luIGFjdGl2YWNpw7NuCiAgICAgICAgICAgIGNvbmZpZGVu'
        'Y2U6ICAgW0IsIExdICAgICAgICAgICDigJQgc2lnbW9pZCDiiIggKDAsIDEpCiAgICAgICAgIiIi'
        'CiAgICAgICAgbm9kZV9zY29yZXMgPSB0b3JjaC5zaWdtb2lkKAogICAgICAgICAgICBzZWxmLm5v'
        'ZGVfc2NvcmVyKGNvbmNlcHRzKS5zcXVlZXplKC0xKQogICAgICAgICkgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbQiwgTF0g4oiIICgwLCAxKQoKICAgICAg'
        'ICB0eXBlX2xvZ2l0cyA9IHNlbGYudHlwZV9jbGFzc2lmaWVyKGNvbmNlcHRzKSAgIyBbQiwgTCwg'
        'bl90eXBlc10KCiAgICAgICAgY29uZmlkZW5jZSA9IHRvcmNoLnNpZ21vaWQoCiAgICAgICAgICAg'
        'IHNlbGYuY29uZmlkZW5jZV9oZWFkKGNvbmNlcHRzKS5zcXVlZXplKC0xKQogICAgICAgICkgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtCLCBMXQoKICAgICAgICBy'
        'ZXR1cm4gbm9kZV9zY29yZXMsIHR5cGVfbG9naXRzLCBjb25maWRlbmNlCg=='
    ),
    'crystallizer/relation_scorer.py': (
        'IiIiCmNyeXN0YWxsaXplci9yZWxhdGlvbl9zY29yZXIucHkg4oCUIEFzeW1tZXRyaWNSZWxhdGlv'
        'blNjb3Jlcgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PQoKUHVudMO6YSByZWxhY2lvbmVzIGRpcmlnaWRhcyBlbnRyZSBwYXJlcyBkZSBu'
        'b2Rvcy4KCkVMIFBST0JMRU1BIENPTiBET1QtUFJPRFVDVCBBVFRFTlRJT046CiAgICBMYSBhdGVu'
        'Y2nDs24gZXN0w6FuZGFyIG1pZGUgU0lNSUxJVFVEOiBzY29yZShBLCBCKSA9IFEoQSnCt0soQikK'
        'ICAgIFBlcm8gUSB5IEsgc29uIGxhIG1pc21hIHByb3llY2Npw7NuIOKGkiBzY29yZShBLEIpID0g'
        'c2NvcmUoQixBKQogICAgTGEgYXRlbmNpw7NuIHNpbcOpdHJpY2EgTk8gcHVlZGUgcmVwcmVzZW50'
        'YXIgIkEgY2F1c2EgQiIg4omgICJCIGNhdXNhIEEiLgoKTEEgU09MVUNJw5NOIOKAlCBQUk9ZRUND'
        'SU9ORVMgQVNJTcOJVFJJQ0FTOgogICAgc291cmNlX3Byb2o6ICLCv3F1w6kgcm9sIGp1ZWdhIGVz'
        'dGUgbm9kbyBDT01PIEZVRU5URSBkZSB1bmEgcmVsYWNpw7NuPyIKICAgIHRhcmdldF9wcm9qOiAi'
        'wr9xdcOpIHJvbCBqdWVnYSBlc3RlIG5vZG8gQ09NTyBERVNUSU5PIGRlIHVuYSByZWxhY2nDs24/'
        'IgoKICAgIHNjb3JlKEHihpJCLCByKSA9IHNvdXJjZV9wcm9qX3IoQSkgwrcgdGFyZ2V0X3Byb2pf'
        'cihCKQogICAgc2NvcmUoQuKGkkEsIHIpID0gc291cmNlX3Byb2pfcihCKSDCtyB0YXJnZXRfcHJv'
        'al9yKEEpCgogICAgQ29tbyBzb3VyY2VfcHJvaiDiiaAgdGFyZ2V0X3Byb2o6CiAgICAgICAgc291'
        'cmNlX3Byb2ooQSkg4omgIHRhcmdldF9wcm9qKEEpICAgZW4gZ2VuZXJhbAogICAg4oaSIHNjb3Jl'
        'KEHihpJCKSDiiaAgc2NvcmUoQuKGkkEpICAgICAgICAgICAgIGVuIGdlbmVyYWwgICDinJMgQVNJ'
        'TcOJVFJJQ08KCiAgICBBbmFsb2fDrWEgZsOtc2ljYToKICAgICAgICBVbmEgZmxlY2hhICjihpIp'
        'IHRpZW5lIHB1bnRhIHkgY29sYSDigJQgbm8gZXMgbG8gbWlzbW8gaW52ZXJ0aXJsYS4KICAgICAg'
        'ICBzb3VyY2VfcHJvaiBjYXB0dXJhIGxhICJwdW50YSIgKGNhdXNhIGFjdGl2YSkuCiAgICAgICAg'
        'dGFyZ2V0X3Byb2ogY2FwdHVyYSBsYSAiY29sYSIgKGVmZWN0byByZWNlcHRpdm8pLgoKSU1QTEVN'
        'RU5UQUNJw5NOOgogICAgUG9yIGVmaWNpZW5jaWEsIHNlIHByb3llY3RhIGEgUsOXSCBkaW1lbnNp'
        'b25lcyBlbiB1biBzb2xvIExpbmVhciwKICAgIGx1ZWdvIHNlIHJlb3JnYW5pemEgY29tbyBSIGNh'
        'YmV6YXMgZGUgSCBkaW1lbnNpb25lcyBjYWRhIHVuYS4KICAgIEVsIHByb2R1Y3RvIGludGVyaW9y'
        'IHBvciBjYWJlemEgZGEgZWwgc2NvcmUgcG9yIHJlbGFjacOzbi4KCiAgICBGaW5hbG1lbnRlLCB1'
        'biBNTFAgcmVmaW5lciBhanVzdGEgbG9zIHNjb3JlcyBjb24gaW5mb3JtYWNpw7NuCiAgICBnbG9i'
        'YWwgZGVsIHZlY3RvciBkZSBzY29yZXMgY29tcGxldG8gKGludGVyYWNjaW9uZXMgZW50cmUgcmVs'
        'YWNpb25lcykuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0'
        'IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFs'
        'bGl6ZXJDb25maWcKCgpjbGFzcyBBc3ltbWV0cmljUmVsYXRpb25TY29yZXIobm4uTW9kdWxlKToK'
        'ICAgICIiIgogICAgUHVudMO6YSBsYXMgUiByZWxhY2lvbmVzIGVudHJlIHRvZG9zIGxvcyBwYXJl'
        'cyAoaeKGkmopIGRlIG5vZG9zLgoKICAgIFVzbzoKICAgICAgICBjZmcgICAgPSBDcnlzdGFsbGl6'
        'ZXJDb25maWcoKQogICAgICAgIHNjb3JlciA9IEFzeW1tZXRyaWNSZWxhdGlvblNjb3JlcihjZmcp'
        'CgogICAgICAgICMgSW5wdXQgYmF0Y2hlZCBbQiwgbiwgRF06CiAgICAgICAgbm9kZXMgID0gdG9y'
        'Y2gucmFuZG4oMiwgOCwgMjU2KQogICAgICAgIGxvZ2l0cyA9IHNjb3Jlcihub2Rlcywgbm9kZXMp'
        'ICAgIyBbMiwgOCwgOCwgMTZdCgogICAgICAgICMgSW5wdXQgc2luIGJhdGNoIFtuLCBEXToKICAg'
        'ICAgICBub2RlcyAgPSB0b3JjaC5yYW5kbig4LCAyNTYpCiAgICAgICAgbG9naXRzID0gc2NvcmVy'
        'KG5vZGVzLCBub2RlcykgICAjIFs4LCA4LCAxNl0KCiAgICBBc2ltZXRyw61hIHZlcmlmaWNhYmxl'
        'OgogICAgICAgIGxvZ2l0c1tpLCBqLCA6XSDiiaAgbG9naXRzW2osIGksIDpdICAoZW4gZ2VuZXJh'
        'bCkKICAgICAgICBwb3JxdWUgc291cmNlX3Byb2og4omgIHRhcmdldF9wcm9qCiAgICAiIiIKCiAg'
        'ICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBDcnlzdGFsbGl6ZXJDb25maWcpIC0+IE5vbmU6'
        'CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGlt'
        'CiAgICAgICAgUiA9IGNvbmZpZy5uX3JlbGF0aW9uX3R5cGVzCiAgICAgICAgSCA9IGNvbmZpZy5y'
        'ZWxhdGlvbl9oaWRkZW5fZGltCgogICAgICAgICMgUHJveWVjY2lvbmVzIGFzaW3DqXRyaWNhczog'
        'bWlzbW8gaW5wdXQgRCwgcm9sZXMgZGlzdGludG9zCiAgICAgICAgIyBDYWRhIHVuYSBwcm9kdWNl'
        'IFIgc3ViLXZlY3RvcmVzIGRlIEggZGltZW5zaW9uZXMKICAgICAgICAjICh1biBzdWItZXNwYWNp'
        'byBwb3IgdGlwbyBkZSByZWxhY2nDs24pCiAgICAgICAgc2VsZi5zb3VyY2VfcHJvaiA9IG5uLkxp'
        'bmVhcihELCBSICogSCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLnRhcmdldF9wcm9qID0gbm4u'
        'TGluZWFyKEQsIFIgKiBILCBiaWFzPUZhbHNlKQoKICAgICAgICAjIFJlZmluYW1pZW50bzogYWp1'
        'c3RhIHNjb3JlcyBmaW5hbGVzIGNvbiBjb250ZXh0byBpbnRlci1yZWxhY2lvbmFsCiAgICAgICAg'
        'IyAoc2FiZXIgcXVlIGhheSBDQVVTRVMgcHVlZGUgbW9kdWxhciBsYSBjZXJ0ZXphIGRlIEVOQUJM'
        'RVMpCiAgICAgICAgc2VsZi5yZWZpbmVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4u'
        'TGluZWFyKFIsIFIgKiAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5M'
        'aW5lYXIoUiAqIDIsIFIpLAogICAgICAgICkKCiAgICAgICAgc2VsZi5fUiA9IFIKICAgICAgICBz'
        'ZWxmLl9IID0gSAoKICAgICAgICBzZWxmLl9pbml0X3dlaWdodHMoKQoKICAgIGRlZiBfaW5pdF93'
        'ZWlnaHRzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIyBQcm95ZWNjaW9uZXM6IGluaXQgbm9ybWFs'
        'IGVzdMOhbmRhcgogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLnNvdXJjZV9wcm9qLndlaWdo'
        'dCwgc3RkPTAuMDIpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYudGFyZ2V0X3Byb2oud2Vp'
        'Z2h0LCBzdGQ9MC4wMikKICAgICAgICAjIFJlZmluZXI6IG5vcm1hbCArIGNlcm9zIGVuIGJpYXNl'
        'cwogICAgICAgIGZvciBtIGluIHNlbGYucmVmaW5lci5tb2R1bGVzKCk6CiAgICAgICAgICAgIGlm'
        'IGlzaW5zdGFuY2UobSwgbm4uTGluZWFyKToKICAgICAgICAgICAgICAgIG5uLmluaXQubm9ybWFs'
        'XyhtLndlaWdodCwgc3RkPTAuMDIpCiAgICAgICAgICAgICAgICBpZiBtLmJpYXMgaXMgbm90IE5v'
        'bmU6CiAgICAgICAgICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obS5iaWFzKQoKICAgIGRlZiBm'
        'b3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgc291cmNlX25vZGVzOiB0b3JjaC5UZW5zb3Is'
        'ICAjIFtCLCBuLCBEXSDDsyBbbiwgRF0KICAgICAgICB0YXJnZXRfbm9kZXM6IHRvcmNoLlRlbnNv'
        'ciwgICMgW0IsIG0sIERdIMOzIFttLCBEXQogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAg'
        'IiIiCiAgICAgICAgQ29tcHV0YSBsb2dpdHMgZGUgcmVsYWNpw7NuIHBhcmEgdG9kb3MgbG9zIHBh'
        'cmVzIGRpcmlnaWRvcyAoaeKGkmopLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBzb3VyY2Vf'
        'bm9kZXM6IFtCLCBuLCBEXSDDsyBbbiwgRF0KICAgICAgICAgICAgdGFyZ2V0X25vZGVzOiBbQiwg'
        'bSwgRF0gw7MgW20sIERdCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIHJlbGF0aW9uX2xv'
        'Z2l0czogW0IsIG4sIG0sIFJdIMOzIFtuLCBtLCBSXQogICAgICAgICAgICAgICAgbG9naXRzWy4u'
        'LiwgaSwgaiwgcl0gPSBzY29yZSBkZSBxdWUgbm9kbyBpIHRpZW5lIHJlbGFjacOzbiByIGNvbiBu'
        'b2RvIGoKICAgICAgICAgICAgICAgIFNpbiBzaWdtb2lkL3NvZnRtYXgg4oCUIGVsIGNvbnN1bWlk'
        'b3IgZGVjaWRlIGxhIGFjdGl2YWNpw7NuLgogICAgICAgICIiIgogICAgICAgIGJhdGNoZWQgPSBz'
        'b3VyY2Vfbm9kZXMuZGltKCkgPT0gMwogICAgICAgIGlmIG5vdCBiYXRjaGVkOgogICAgICAgICAg'
        'ICBzb3VyY2Vfbm9kZXMgPSBzb3VyY2Vfbm9kZXMudW5zcXVlZXplKDApICAjIFsxLCBuLCBEXQog'
        'ICAgICAgICAgICB0YXJnZXRfbm9kZXMgPSB0YXJnZXRfbm9kZXMudW5zcXVlZXplKDApICAjIFsx'
        'LCBtLCBEXQoKICAgICAgICBCLCBuLCBEID0gc291cmNlX25vZGVzLnNoYXBlCiAgICAgICAgbSA9'
        'IHRhcmdldF9ub2Rlcy5zaGFwZVsxXQogICAgICAgIFIsIEggPSBzZWxmLl9SLCBzZWxmLl9ICgog'
        'ICAgICAgICMgW0IsIG4sIFIsIEhdIOKAlCBjYWRhIG5vZG8gcHJveWVjdGFkbyBlbiBzdSByb2wg'
        'ZGUgRlVFTlRFCiAgICAgICAgcyA9IHNlbGYuc291cmNlX3Byb2ooc291cmNlX25vZGVzKS52aWV3'
        'KEIsIG4sIFIsIEgpCgogICAgICAgICMgW0IsIG0sIFIsIEhdIOKAlCBjYWRhIG5vZG8gcHJveWVj'
        'dGFkbyBlbiBzdSByb2wgZGUgREVTVElOTwogICAgICAgIHQgPSBzZWxmLnRhcmdldF9wcm9qKHRh'
        'cmdldF9ub2RlcykudmlldyhCLCBtLCBSLCBIKQoKICAgICAgICAjIHNjb3JlKGnihpJqLCByKSA9'
        'IGRvdChzW2IsaSxyLDpdLCB0W2IsaixyLDpdKQogICAgICAgICMgW0IsIG4sIG0sIFJdID0gZWlu'
        'c3VtIHNvYnJlIGxhIGRpbWVuc2nDs24gSAogICAgICAgIHNjb3JlcyA9IHRvcmNoLmVpbnN1bSgi'
        'Ym5yaCxibXJoLT5ibm1yIiwgcywgdCkKCiAgICAgICAgIyBSZWZpbmFtaWVudG8gcG9zdC1wdW50'
        'dWFjacOzbgogICAgICAgIHJlZmluZWQgPSBzZWxmLnJlZmluZXIoc2NvcmVzKSAgIyBbQiwgbiwg'
        'bSwgUl0KCiAgICAgICAgcmV0dXJuIHJlZmluZWQgaWYgYmF0Y2hlZCBlbHNlIHJlZmluZWQuc3F1'
        'ZWV6ZSgwKSAgIyBbbiwgbSwgUl0K'
    ),
    'crystallizer/pooler.py': (
        'IiIiCmNyeXN0YWxsaXplci9wb29sZXIucHkg4oCUIENyb3NzQXR0ZW50aW9uUG9vbGVyCj09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKQWdyZWdhIGluZm9ybWFj'
        'acOzbiBkZWwgY29udGV4dG8gY29tcGxldG8gZW4gY2FkYSB2ZWN0b3IgZGUgbm9kbyBkZXRlY3Rh'
        'ZG8uCgpQUk9CTEVNQSBRVUUgUkVTVUVMVkU6CiAgICBFbCBTdHJlYW1FbmNvZGVyIHlhIGxlIGFz'
        'aWduw7MgdW4gdmVjdG9yIGEgY2FkYSBwb3NpY2nDs24uCiAgICBQZXJvIGVzZSB2ZWN0b3Igc29s'
        'byBjYXB0dXJhIGVsIGNvbnRleHRvIENBVVNBTCAodG9rZW5zIGFudGVyaW9yZXMsIGdyYWNpYXMg'
        'YWwgU1NNKS4KICAgIEVsIEdyYXBoQ3J5c3RhbGxpemVyIG5lY2VzaXRhIHVuIHZlY3RvciBxdWUg'
        'Y2FwdHVyZSBlbCBDT05URVhUTyBDT01QTEVUTyBkZWwgbm9kbwogICAg4oCUIHRhbnRvIGxvIHF1'
        'ZSBwcmVjZWRlIGNvbW8gbG8gcXVlIHNpZ3VlIOKAlCBwYXJhIGRlY2lkaXIgbGEgcmVsYWNpw7Nu'
        'IGNvbiBvdHJvcyBub2Rvcy4KCiAgICBFamVtcGxvOiAiRWwgZnVlZ28gW05PRE9dIHNlIGV4dGlu'
        'Z3Vpw7MiIHZzICJFbCBmdWVnbyBbTk9ET10gc2UgcHJvcGFnw7MiCiAgICBFbCBzaWduaWZpY2Fk'
        'byBkZSAiZnVlZ28iIGNvbW8gbm9kbyBkZWwgZ3JhZm8gZGVwZW5kZSBkZWwgY29udGV4dG8gYmlk'
        'aXJlY2Npb25hbC4KClNPTFVDScOTTjoKICAgIENyb3NzLWF0dGVudGlvbiBkb25kZToKICAgICAg'
        'ICBRdWVyaWVzICA9IHZlY3RvcmVzIGRlIHBvc2ljaW9uZXMgZGV0ZWN0YWRhcyBjb21vIG5vZG9z'
        'IChsbyBxdWUgcXVlcmVtb3MgZW5yaXF1ZWNlcikKICAgICAgICBLZXlzICAgICA9IHNlY3VlbmNp'
        'YSBjb21wbGV0YSBkZSBjb25jZXB0IHZlY3RvcnMgKGNvbnRleHRvIGEgY29uc3VsdGFyKQogICAg'
        'ICAgIFZhbHVlcyAgID0gc2VjdWVuY2lhIGNvbXBsZXRhIGRlIGNvbmNlcHQgdmVjdG9ycyAoaW5m'
        'b3JtYWNpw7NuIGEgYWdyZWdhcikKCiAgICBSZXN1bHRhZG86IGNhZGEgbm9kbyB0aWVuZSB1biB2'
        'ZWN0b3IgZW5yaXF1ZWNpZG8gY29uIGluZm9ybWFjacOzbiBnbG9iYWwuCiAgICBMb3MgdmVjdG9y'
        'ZXMgZGUgbm9kbyBzb24gbGEgaW5wdXQgYWwgQXN5bW1ldHJpY1JlbGF0aW9uU2NvcmVyLgoKUE9S'
        'IFFVw4kgQ1JPU1MtQVRURU5USU9OIFkgTk8gU0VMRi1BVFRFTlRJT046CiAgICBTZWxmLWF0dGVu'
        'dGlvbiBlbnRyZSBub2RvcyByZXF1ZXJpcsOtYSB5YSB0ZW5lciBsb3MgdmVjdG9yZXMgZmluYWxl'
        'cyBkZSBub2RvLgogICAgQ3Jvc3MtYXR0ZW50aW9uIGNvbiBlbCBjb250ZXh0byBjb21wbGV0byBl'
        'cyBlbCBwYXNvIHF1ZSBjb25zdHJ1eWUgZXNvcyB2ZWN0b3Jlcy4KIiIiCgpmcm9tIF9fZnV0dXJl'
        'X18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0'
        'b3JjaC5ubiBhcyBubgoKZnJvbSAuY29uZmlnIGltcG9ydCBDcnlzdGFsbGl6ZXJDb25maWcKCgpj'
        'bGFzcyBDcm9zc0F0dGVudGlvblBvb2xlcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBBZ3JlZ2Eg'
        'Y29udGV4dG8gZW4gdmVjdG9yZXMgZGUgbm9kbyBtZWRpYW50ZSBjcm9zcy1hdHRlbnRpb24uCgog'
        'ICAgVXNvOgogICAgICAgIGNmZyAgICA9IENyeXN0YWxsaXplckNvbmZpZygpCiAgICAgICAgcG9v'
        'bGVyID0gQ3Jvc3NBdHRlbnRpb25Qb29sZXIoY2ZnKQogICAgICAgIHF1ZXJpZXMgPSB0b3JjaC5y'
        'YW5kbigyLCA4LCAyNTYpICAgIyBbQiwgbl9ub2RlcywgRF0KICAgICAgICBjb250ZXh0ID0gdG9y'
        'Y2gucmFuZG4oMiwgNjQsIDI1NikgICMgW0IsIEwsIERdCiAgICAgICAgb3V0ID0gcG9vbGVyKHF1'
        'ZXJpZXMsIGNvbnRleHQpICAgICAjIFtCLCBuX25vZGVzLCBEXQogICAgIiIiCgogICAgZGVmIF9f'
        'aW5pdF9fKHNlbGYsIGNvbmZpZzogQ3J5c3RhbGxpemVyQ29uZmlnKSAtPiBOb25lOgogICAgICAg'
        'IHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIEQgPSBjb25maWcuaGlkZGVuX2RpbQogICAgICAg'
        'IEggPSBjb25maWcucG9vbGVyX2hlYWRzCgogICAgICAgIHNlbGYubl9oZWFkcyA9IEgKICAgICAg'
        'ICBzZWxmLmhlYWRfZGltID0gRCAvLyBICiAgICAgICAgc2VsZi5zY2FsZSA9IG1hdGguc3FydChz'
        'ZWxmLmhlYWRfZGltKSAqKiAtMSAgIyA9IDEv4oiaaGVhZF9kaW0KCiAgICAgICAgIyBRdWVyaWVz'
        'IHZpZW5lbiBkZSBsb3Mgbm9kb3MgZGV0ZWN0YWRvcwogICAgICAgICMgS2V5cyB5IFZhbHVlcyB2'
        'aWVuZW4gZGVsIGNvbnRleHRvIGNvbXBsZXRvCiAgICAgICAgc2VsZi5xX3Byb2ogICA9IG5uLkxp'
        'bmVhcihELCBELCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYua19wcm9qICAgPSBubi5MaW5lYXIo'
        'RCwgRCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLnZfcHJvaiAgID0gbm4uTGluZWFyKEQsIEQs'
        'IGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5vdXRfcHJvaiAgPSBubi5MaW5lYXIoRCwgRCwgYmlh'
        'cz1GYWxzZSkKCiAgICAgICAgIyBOb3JtYWxpemFjacOzbiBwb3N0LXJlc2lkdWFsIChlc3TDoW5k'
        'YXIgZW4gYmxvcXVlcyBkZSBjcm9zcy1hdHRlbnRpb24pCiAgICAgICAgc2VsZi5ub3JtID0gbm4u'
        'TGF5ZXJOb3JtKEQpCgogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0'
        'X3dlaWdodHMoc2VsZikgLT4gTm9uZToKICAgICAgICAjIFByb3llY2Npb25lcyBjb24gc3RkIG1h'
        'eW9yIHF1ZSAwLjAyIHBhcmEgcXVlIGxhIGNyb3NzLWF0dGVudGlvbgogICAgICAgICMgcHJvZHV6'
        'Y2EgcmVwcmVzZW50YWNpb25lcyBkaXN0aW50YXMgcG9yIG5vZG8gZGVzZGUgZWwgaW5pY2lvLgog'
        'ICAgICAgICMgQ29uIHN0ZD0wLjAyIChkZWZhdWx0IExMTSksIFHiiYgwIOKGkiBhdGVuY2nDs24g'
        'dW5pZm9ybWUg4oaSIHRvZG9zIGxvcyBub2RvcwogICAgICAgICMgcmVjaWJlbiBlbCBtaXNtbyB2'
        'ZWN0b3IgYWdyZWdhZG8sIHBlcmRpZW5kbyBsYSBpZGVudGlkYWQgZGVsIG5vZG8uCiAgICAgICAg'
        'IyBzdGQ9MC4xIHByb2R1Y2Ugc2NvcmVzIGRlIGF0ZW5jacOzbiBjb24gdmFyaWFuemEgc3VmaWNp'
        'ZW50ZSBwYXJhIGRpZmVyZW5jaWFybG9zLgogICAgICAgIHN0ZCA9IDAuMQogICAgICAgIGZvciBw'
        'cm9qIGluIChzZWxmLnFfcHJvaiwgc2VsZi5rX3Byb2osIHNlbGYudl9wcm9qKToKICAgICAgICAg'
        'ICAgbm4uaW5pdC5ub3JtYWxfKHByb2oud2VpZ2h0LCBzdGQ9c3RkKQogICAgICAgICMgb3V0X3By'
        'b2ogbcOhcyBjb25zZXJ2YWRvcmEgcGFyYSBlc3RhYmlsaWRhZCBkZWwgcmVzaWR1YWwKICAgICAg'
        'ICBubi5pbml0Lm5vcm1hbF8oc2VsZi5vdXRfcHJvai53ZWlnaHQsIHN0ZD0wLjAyKQoKICAgIGRl'
        'ZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgbm9kZV9xdWVyaWVzOiB0b3JjaC5UZW5z'
        'b3IsICAjIFtCLCBuX25vZGVzLCBEXQogICAgICAgIGNvbnRleHQ6IHRvcmNoLlRlbnNvciwgICAg'
        'ICAgIyBbQiwgTCwgRF0KICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgogICAgICAg'
        'IEFyZ3M6CiAgICAgICAgICAgIG5vZGVfcXVlcmllczogW0IsIG5fbm9kZXMsIERdIOKAlCB2ZWN0'
        'b3JlcyBkZSBub2RvIChxdWVyaWVzKQogICAgICAgICAgICBjb250ZXh0OiAgICAgIFtCLCBMLCBE'
        'XSAgICAgICDigJQgc2VjdWVuY2lhIGNvbXBsZXRhIChrZXlzICsgdmFsdWVzKQoKICAgICAgICBS'
        'ZXR1cm5zOgogICAgICAgICAgICBub2RlX3ZlY3RvcnM6IFtCLCBuX25vZGVzLCBEXSDigJQgbm9k'
        'b3MgZW5yaXF1ZWNpZG9zIGNvbiBjb250ZXh0bwogICAgICAgICIiIgogICAgICAgIEIsIG4sIEQg'
        'PSBub2RlX3F1ZXJpZXMuc2hhcGUKICAgICAgICBMID0gY29udGV4dC5zaGFwZVsxXQogICAgICAg'
        'IEggPSBzZWxmLm5faGVhZHMKICAgICAgICBIZCA9IHNlbGYuaGVhZF9kaW0KCiAgICAgICAgIyBQ'
        'cm95ZWN0YXIgeSBzZXBhcmFyIGNhYmV6YXMKICAgICAgICBRID0gc2VsZi5xX3Byb2oobm9kZV9x'
        'dWVyaWVzKS52aWV3KEIsIG4sIEgsIEhkKS50cmFuc3Bvc2UoMSwgMikgICMgW0IsIEgsIG4sIEhk'
        'XQogICAgICAgIEsgPSBzZWxmLmtfcHJvaihjb250ZXh0KS52aWV3KEIsIEwsIEgsIEhkKS50cmFu'
        'c3Bvc2UoMSwgMikgICAgICAgICMgW0IsIEgsIEwsIEhkXQogICAgICAgIFYgPSBzZWxmLnZfcHJv'
        'aihjb250ZXh0KS52aWV3KEIsIEwsIEgsIEhkKS50cmFuc3Bvc2UoMSwgMikgICAgICAgICMgW0Is'
        'IEgsIEwsIEhkXQoKICAgICAgICAjIEF0ZW5jacOzbjogY2FkYSBub2RvIGNvbnN1bHRhIHRvZGEg'
        'bGEgc2VjdWVuY2lhCiAgICAgICAgYXR0bl93ZWlnaHRzID0gKFEgQCBLLnRyYW5zcG9zZSgtMiwg'
        'LTEpKSAqIHNlbGYuc2NhbGUgICMgW0IsIEgsIG4sIExdCiAgICAgICAgYXR0bl93ZWlnaHRzID0g'
        'dG9yY2guc29mdG1heChhdHRuX3dlaWdodHMsIGRpbT0tMSkgICAgICMgW0IsIEgsIG4sIExdCgog'
        'ICAgICAgICMgQWdyZWdhY2nDs24KICAgICAgICBhdHRuX291dCA9IGF0dG5fd2VpZ2h0cyBAIFYg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgW0IsIEgsIG4sIEhkXQogICAg'
        'ICAgIGF0dG5fb3V0ID0gYXR0bl9vdXQudHJhbnNwb3NlKDEsIDIpLmNvbnRpZ3VvdXMoKS52aWV3'
        'KEIsIG4sIEQpICAjIFtCLCBuLCBEXQoKICAgICAgICAjIFJlc2lkdWFsOiBjYWRhIG5vZG8gcmV0'
        'aWVuZSBzdSB2ZWN0b3IgZGUgY29uc3VsdGEgb3JpZ2luYWwuCiAgICAgICAgIyBFc3RvIGdhcmFu'
        'dGl6YSBxdWUgbm9kb3MgZGlzdGludG9zIHByb2R1emNhbiByZXByZXNlbnRhY2lvbmVzIGRpc3Rp'
        'bnRhcywKICAgICAgICAjIGluY2x1c28gY3VhbmRvIGxhIGNyb3NzLWF0dGVudGlvbiBlc3TDoSBp'
        'bmljaWFsaXphZGEgY29uIHBlc29zIHBlcXVlw7Fvcy4KICAgICAgICAjIG5vZGVfdmVjdG9yc1tp'
        'XSA9IExOKG5vZGVfcXVlcmllc1tpXSArIG91dF9wcm9qKGF0dG5fb3V0W2ldKSkKICAgICAgICBy'
        'ZXR1cm4gc2VsZi5ub3JtKG5vZGVfcXVlcmllcyArIHNlbGYub3V0X3Byb2ooYXR0bl9vdXQpKSAg'
        'ICAgICAgICMgW0IsIG4sIERdCg=='
    ),
    'crystallizer/model.py': (
        '...'
        'ICAgIH0K'
    ),
    'cre/__init__.py': (
        'IiIiCkFJT04tQyBjcmUg4oCUIENhdXNhbCBSZWFzb25pbmcgRW5naW5lLgoKTW90b3IgZGUgcmF6'
        'b25hbWllbnRvIGl0ZXJhdGl2byBiYXNhZG8gZW4gdHlwZWQgbWVzc2FnZSBwYXNzaW5nIGNvbiB3'
        'ZWlnaHQgc2hhcmluZy4KQ2F1c2FsR3JhcGggKyBub2RlIGZlYXR1cmVzIOKGkiBub2RlIGZlYXR1'
        'cmVzIHJlZmluYWRvcy4KCkNvbXBvbmVudGVzIGFkaWNpb25hbGVzIChwYXJhZGEgYWRhcHRhdGl2'
        'YSk6CiAgICBXZWFrbmVzc0RldGVjdG9yICDigJQgZGV0ZWN0YSBkZWJpbGlkYWRlcyBlbiBlbCBn'
        'cmFmbyBwYXJhIGd1aWFyIHJlZmluYW1pZW50bwogICAgQ29udmVyZ2VuY2VHYXRlICAg4oCUIGRl'
        'Y2lkZSBjdcOhbmRvIHBhcmFyIGRlIGl0ZXJhciAoYWN0aXZvIGNvbiB1c2VfY29udmVyZ2VuY2Vf'
        'Z2F0ZT1UcnVlKQoKQ29tcG9uZW50ZXMgb3BjaW9uYWxlcyAoY2FwYWNpZGFkKToKICAgIFNwYXJz'
        'ZU1vRSAvIEV4cGVydEdyb3VwIOKAlCBlc3BlY2lhbGl6YWNpw7NuIHBvc3QtTVAgKGFjdGl2byBj'
        'b24gdXNlX21vZT1UcnVlKQoiIiIKCmZyb20gLmFnZ3JlZ2F0b3IgaW1wb3J0IEF0dGVudGl2ZUFn'
        'Z3JlZ2F0b3IKZnJvbSAuYXV0b19zY2FsZSBpbXBvcnQgQXV0b1NjYWxlciwgQXV0b1NjYWxlUmVz'
        'dWx0CmZyb20gLmJhdGNoaW5nIGltcG9ydCBCYXRjaGVkR3JhcGgsIFB5R1N0eWxlQmF0Y2hlcgpm'
        'cm9tIC5jb25maWcgaW1wb3J0IENSRUNvbmZpZwpmcm9tIC5jb252ZXJnZW5jZSBpbXBvcnQgQ29u'
        'dmVyZ2VuY2VEZWNpc2lvbiwgQ29udmVyZ2VuY2VHYXRlCmZyb20gLmVuZ2luZSBpbXBvcnQgQ1JF'
        'T3V0cHV0LCBDYXVzYWxSZWFzb25pbmdFbmdpbmUKZnJvbSAubWVzc2FnZV9wYXNzaW5nIGltcG9y'
        'dCBDYXVzYWxNZXNzYWdlUGFzc2luZ0xheWVyCmZyb20gLm1vZSBpbXBvcnQgRXhwZXJ0R3JvdXAs'
        'IE1vRU91dHB1dCwgU3BhcnNlTW9FCmZyb20gLnNjcmF0Y2hfcGFkIGltcG9ydCBEaWZmZXJlbnRp'
        'YWJsZVNjcmF0Y2hQYWQsIFNjcmF0Y2hQYWRDb25maWcKZnJvbSAud2Vha25lc3MgaW1wb3J0ICgK'
        'ICAgIFdFQUtORVNTX1RZUEVTLAogICAgV2Vha25lc3MsCiAgICBXZWFrbmVzc0RldGVjdG9yLAog'
        'ICAgV2Vha25lc3NSZXBvcnQsCikKCl9fYWxsX18gPSBbCiAgICAiQXR0ZW50aXZlQWdncmVnYXRv'
        'ciIsCiAgICAiQXV0b1NjYWxlUmVzdWx0IiwKICAgICJBdXRvU2NhbGVyIiwKICAgICJCYXRjaGVk'
        'R3JhcGgiLAogICAgIkNSRUNvbmZpZyIsCiAgICAiQ1JFT3V0cHV0IiwKICAgICJDYXVzYWxNZXNz'
        'YWdlUGFzc2luZ0xheWVyIiwKICAgICJDYXVzYWxSZWFzb25pbmdFbmdpbmUiLAogICAgIkNvbnZl'
        'cmdlbmNlRGVjaXNpb24iLAogICAgIkNvbnZlcmdlbmNlR2F0ZSIsCiAgICAiRGlmZmVyZW50aWFi'
        'bGVTY3JhdGNoUGFkIiwKICAgICJFeHBlcnRHcm91cCIsCiAgICAiTW9FT3V0cHV0IiwKICAgICJQ'
        'eUdTdHlsZUJhdGNoZXIiLAogICAgIlNjcmF0Y2hQYWRDb25maWciLAogICAgIlNwYXJzZU1vRSIs'
        'CiAgICAiV0VBS05FU1NfVFlQRVMiLAogICAgIldlYWtuZXNzIiwKICAgICJXZWFrbmVzc0RldGVj'
        'dG9yIiwKICAgICJXZWFrbmVzc1JlcG9ydCIsCl0K'
    ),
    'cre/config.py': (
        'IiIiCmNyZS9jb25maWcucHkg4oCUIENvbmZpZ3VyYWNpw7NuIGRlbCBDYXVzYWwgUmVhc29uaW5n'
        'IEVuZ2luZQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBk'
        'YXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcwpjbGFzcyBDUkVDb25maWc6'
        'CiAgICAiIiIKICAgIEhpcGVycGFyw6FtZXRyb3MgZGVsIENhdXNhbFJlYXNvbmluZ0VuZ2luZS4K'
        'CiAgICBDb25maWd1cmFjacOzbiB0aW55IHBhcmEgdGVzdGluZzoKICAgICAgICBub2RlX2RpbT0y'
        'NTYsIGVkZ2VfZGltPTY0LCBtZXNzYWdlX2RpbT0xMjgsCiAgICAgICAgbl9tZXNzYWdlX2xheWVy'
        'cz0yLCBtYXhfaXRlcmF0aW9ucz0yMAoKICAgIFdFSUdIVCBTSEFSSU5HOgogICAgICAgIExvcyBu'
        'X21lc3NhZ2VfbGF5ZXJzIGNhcGFzIHNlIGNvbXBhcnRlbiBlbnRyZSBUT0RBUyBsYXMgaXRlcmFj'
        'aW9uZXMuCiAgICAgICAgVG90YWwgZGUgcGFyw6FtZXRyb3MgPSBwYXLDoW1ldHJvcyBkZSAyIGNh'
        'cGFzIChubyBkZSAyIMOXIDIwID0gNDAgY2FwYXMpLgogICAgICAgIFRvdGFsIGRlIEZMT1BzID0g'
        'MiBjYXBhcyDDlyBtYXhfaXRlcmF0aW9ucyAobcOhcyBiYXJhdG8gcXVlIDQwIGNhcGFzIGRpc3Rp'
        'bnRhcykuCgogICAgbl9yZWxhdGlvbl90eXBlczoKICAgICAgICBEZWJlIGNvaW5jaWRpciBjb24g'
        'bGVuKENBVVNBTF9SRUxBVElPTlMpID0gMTYuCiAgICAgICAgQ2FkYSByZWxhY2nDs24gdGllbmUg'
        'c3UgcHJvcGlhIGZ1bmNpw7NuIGRlIG1lbnNhamUgZW4gQ2F1c2FsTWVzc2FnZVBhc3NpbmdMYXll'
        'ci4KICAgICIiIgoKICAgICMgRGltZW5zaW9uZXMgcHJpbmNpcGFsZXMKICAgIG5vZGVfZGltOiAg'
        'ICAgIGludCA9IDI1NiAgICMgZGltZW5zacOzbiBkZSBsb3MgdmVjdG9yZXMgZGUgbm9kbwogICAg'
        'ZWRnZV9kaW06ICAgICAgaW50ID0gNjQgICAgIyBkaW1lbnNpw7NuIGRlIGxvcyB2ZWN0b3JlcyBk'
        'ZSBhcmlzdGEKICAgIG1lc3NhZ2VfZGltOiAgIGludCA9IDEyOCAgICMgZGltZW5zacOzbiBkZSBs'
        'b3MgbWVuc2FqZXMgZW50cmUgbm9kb3MKCiAgICAjIEFycXVpdGVjdHVyYQogICAgbl9tZXNzYWdl'
        'X2xheWVyczogaW50ID0gMiAgIyBjYXBhcyBkZSBNUCBwb3IgaXRlcmFjacOzbiAoY29tcGFydGlk'
        'YXMpCiAgICBtYXhfaXRlcmF0aW9uczogICBpbnQgPSAyMCAgIyBpdGVyYWNpb25lcyBkZSByZWZp'
        'bmFtaWVudG8gcG9yIGRlZmVjdG8KCiAgICAjIFZvY2FidWxhcmlvIChkZWJlIGNvaW5jaWRpciBj'
        'b24gY29yZS9ncmFwaC5weSkKICAgIG5fcmVsYXRpb25fdHlwZXM6IGludCA9IDE2ICAjIGxlbihD'
        'YXVzYWxSZWxhdGlvbikKCiAgICAjIEVzdGFiaWxpZGFkIG51bcOpcmljYQogICAgbm9ybV9lcHM6'
        'IGZsb2F0ID0gMWUtNgoKICAgICMg4pSA4pSAIENvbnZlcmdlbmNlR2F0ZSAocGFyYWRhIGRpbsOh'
        'bWljYSkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        'CiAgICAjIHVzZV9jb252ZXJnZW5jZV9nYXRlPUZhbHNlIHByZXNlcnZhIGVsIGNvbXBvcnRhbWll'
        'bnRvIGFudGVyaW9yIChpdGVyYWNpb25lcyBmaWphcykuCiAgICAjIEFjdGl2YXIgcGFyYSBiZW5l'
        'ZmljaWFyc2UgZGUgcGFyYWRhIGFkYXB0YXRpdmEuCiAgICB1c2VfY29udmVyZ2VuY2VfZ2F0ZTog'
        'IGJvb2wgID0gVHJ1ZSAgICMgYWN0aXZhciBwYXJhZGEgZGluw6FtaWNhCiAgICBtaW5faXRlcmF0'
        'aW9uczogICAgICAgIGludCAgID0gMSAgICAgICMgc2FmZXR5IGZsb29yOiBudW5jYSBwYXJhciBh'
        'bnRlcyBkZSBlc3RvCgogICAgIyBUaHJlc2hvbGRzIGRlbCBDb252ZXJnZW5jZUdhdGUKICAgIGNv'
        'bnZfZGVsdGFfdGhyZXNob2xkOiAgIGZsb2F0ID0gMC4wMSAgICMgfHzOlGh8fC98fGh8fCA8IGVz'
        'dG8g4oaSIGZlYXR1cmVzIGVzdGFibGVzCiAgICBjb252X2NvbmZfdGhyZXNob2xkOiAgICBmbG9h'
        'dCA9IDAuNzUgICAjIGNvbmZpYW56YSBtZWRpYSA+IGVzdG8g4oaSIG5vZG9zIHNlZ3Vyb3MKICAg'
        'IGNvbnZfd2Vha25lc3NfdGhyZXNob2xkOiBmbG9hdCA9IDAuMjUgICMgcmF0aW8gZGViaWxpZGFk'
        'ZXMgPCBlc3RvIOKGkiBtYXlvcsOtYSByZXN1ZWx0YQoKICAgICMg4pSA4pSAIFdlYWtuZXNzRGV0'
        'ZWN0b3Ig4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAj'
        'IFPDs2xvIHNlIGluc3RhbmNpYSBjdWFuZG8gdXNlX2NvbnZlcmdlbmNlX2dhdGU9VHJ1ZS4KICAg'
        'IHdlYWtuZXNzX2NvbmZfdGhyZXNob2xkOiBmbG9hdCA9IDAuMzUgICMgc2lnbW9pZCA8IGVzdG8g'
        '4oaSIGxvd19jb25maWRlbmNlCiAgICB3ZWFrbmVzc19jb25mX2hpZGRlbjogICAgaW50ICAgPSA2'
        'NCAgICAjIGRpbSBvY3VsdGEgZGVsIGNvbmZpZGVuY2Ugc2NvcmVyCgogICAgIyDilIDilIAgU3Bh'
        'cnNlTW9FIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgAogICAgIyB1c2VfbW9lPUZhbHNlIHByZXNlcnZhIGVsIGNvbXBvcnRh'
        'bWllbnRvIGFudGVyaW9yIChzaW4gTW9FKS4KICAgICMgRWwgTW9FIGFjdMO6YSBERVNQVcOJUyBk'
        'ZWwgbWVzc2FnZSBwYXNzaW5nIHBvciBpdGVyYWNpw7NuLgogICAgdXNlX21vZTogICAgICAgICAg'
        'ICAgICBib29sICA9IEZhbHNlICAjIGFjdGl2YXIgU3BhcnNlTW9FIHBvc3QtTVAKICAgIG1vZV9u'
        'X2dyb3VwczogICAgICAgICAgaW50ICAgPSA0ICAgICAgIyBncnVwb3MgZGUgZXhwZXJ0b3MKICAg'
        'IG1vZV9leHBlcnRzX3Blcl9ncm91cDogaW50ICAgPSA0ICAgICAgIyBleHBlcnRvcyBwb3IgZ3J1'
        'cG8gKHRvdGFsID0gbl9ncm91cHMgw5cgcGVyX2dyb3VwKQogICAgbW9lX2FjdGl2ZV9leHBlcnRz'
        'OiAgICBpbnQgICA9IDIgICAgICAjIHRvcC1rIGFjdGl2b3MgcG9yIG5vZG8KICAgIG1vZV9leHBl'
        'cnRfaGlkZGVuX211bHQ6IGludCAgPSAyICAgICAgIyBtdWx0aXBsaWNhZG9yIG9jdWx0byBkZW50'
        'cm8gZGUgY2FkYSBleHBlcnRvCiAgICBtb2VfbG9hZF9iYWxhbmNlX3dlaWdodDogZmxvYXQgPSAw'
        'LjAxICAjIHBlc28gZGUgbGEgbG9hZCBiYWxhbmNlIGxvc3MKCiAgICBkZWYgX19wb3N0X2luaXRf'
        'XyhzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYubl9tZXNzYWdlX2xheWVycyA8IDE6CiAg'
        'ICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJuX21lc3NhZ2VfbGF5ZXJzIG11c3QgYmUgPj0g'
        'MSwgZ290IHtzZWxmLm5fbWVzc2FnZV9sYXllcnN9IikKICAgICAgICBpZiBzZWxmLm1heF9pdGVy'
        'YXRpb25zIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1heF9pdGVyYXRpb25z'
        'IG11c3QgYmUgPj0gMSwgZ290IHtzZWxmLm1heF9pdGVyYXRpb25zfSIpCiAgICAgICAgaWYgc2Vs'
        'Zi5tZXNzYWdlX2RpbSA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJtZXNzYWdl'
        'X2RpbSBtdXN0IGJlID49IDEsIGdvdCB7c2VsZi5tZXNzYWdlX2RpbX0iKQogICAgICAgIGlmIHNl'
        'bGYubWluX2l0ZXJhdGlvbnMgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibWlu'
        'X2l0ZXJhdGlvbnMgbXVzdCBiZSA+PSAxLCBnb3Qge3NlbGYubWluX2l0ZXJhdGlvbnN9IikKICAg'
        'ICAgICBpZiBzZWxmLm1pbl9pdGVyYXRpb25zID4gc2VsZi5tYXhfaXRlcmF0aW9uczoKICAgICAg'
        'ICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYibWluX2l0ZXJhdGlvbnMg'
        'KHtzZWxmLm1pbl9pdGVyYXRpb25zfSkgPiBtYXhfaXRlcmF0aW9ucyAoe3NlbGYubWF4X2l0ZXJh'
        'dGlvbnN9KSIKICAgICAgICAgICAgKQo='
    ),
    'cre/aggregator.py': (
        'IiIiCmNyZS9hZ2dyZWdhdG9yLnB5IOKAlCBBdHRlbnRpdmVBZ2dyZWdhdG9yCj09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCkFncmVnYSBtZW5zYWplcyBlbnRyYW50ZXMg'
        'ZW4gY2FkYSBub2RvLCBwb25kZXJhbmRvIHBvciBpbXBvcnRhbmNpYSBhcHJlbmRpZGEuCgpQT1Ig'
        'UVXDiSBBVFRFTlRJVkUgWSBOTyBNRUFOOgogICAgTWVhbiBhZ2dyZWdhdGlvbiB0cmF0YSB0b2Rv'
        'cyBsb3MgbWVuc2FqZXMgY29tbyBpZ3VhbG1lbnRlIGltcG9ydGFudGVzLgogICAgUGVybyBlbiBy'
        'YXpvbmFtaWVudG8gY2F1c2FsOgogICAgICAgIC0gVW4gbWVuc2FqZSBkZSBDQVVTRVMgZnVlcnRl'
        'IGVzIG3DoXMgcmVsZXZhbnRlIHF1ZSB1bm8gZGUgQ09SUkVMQVRFUyBkw6liaWwKICAgICAgICAt'
        'IFVuIG1lbnNhamUgcXVlIGNvbmZpcm1hIGVsIGVzdGFkbyBhY3R1YWwgZXMgbWVub3MgdXJnZW50'
        'ZSBxdWUgdW5vIHF1ZSBsbyBjb250cmFkaWNlCiAgICAgICAgLSBFbCBub2RvIGRlc3Rpbm8gZGVi'
        'ZXLDrWEgcG9kZXIgImZpbHRyYXIiIG1lbnNhamVzIGlycmVsZXZhbnRlcwoKICAgIEVsIEF0dGVu'
        'dGl2ZUFnZ3JlZ2F0b3IgYXByZW5kZSBhIGFzaWduYXIgdW4gcGVzbyDPgyDiiIggKDAsMSkgYSBj'
        'YWRhIG1lbnNhamUKICAgIGJhc2FkbyBlbiAoY29udGVuaWRvIGRlbCBtZW5zYWplLCBlc3RhZG8g'
        'YWN0dWFsIGRlbCBub2RvIGRlc3Rpbm8pLgoKSU1QTEVNRU5UQUNJw5NOOgogICAgUGFyYSBjYWRh'
        'IG1lbnNhamUgZW50cmFudGU6CiAgICAgICAgc2NvcmUgPSBzaWdtb2lkKE1MUChbbWVuc2FqZSwg'
        'ZXN0YWRvX25vZG9fZGVzdGlub10pKQogICAgICAgIHBlc28gID0gc2NvcmUgIChubyBzb2Z0bWF4'
        'IOKAlCB2ZXIgbm90YSBhYmFqbykKCiAgICBMdWVnbzogYWdyZWdhZG8gPSBzdW0ocGVzbyDDlyBt'
        'ZW5zYWplKSAvIChzdW0ocGVzbykgKyDOtSkKCiAgICBQb3IgcXXDqSBzaWdtb2lkIHkgbm8gc29m'
        'dG1heDoKICAgICAgICBTb2Z0bWF4IG5vcm1hbGl6YSBzb2JyZSBsb3MgbWVuc2FqZXMgZGUgVU4g'
        'bm9kbyDihpIgcmVxdWllcmUgcGVyLXRhcmdldCBzZWdtZW50IHNvZnRtYXgKICAgICAgICAobm8g'
        'ZGlzcG9uaWJsZSBkaXJlY3RhbWVudGUgZW4gUHlUb3JjaCBzaW4gdG9yY2hfc2NhdHRlcikuCiAg'
        'ICAgICAgU2lnbW9pZCBkYSBwZXNvcyBpbmRlcGVuZGllbnRlcyDihpIgbGEgc3VtYSBub3JtYWxp'
        'emFkYSBwb3IgY291bnQvc3VtX3dlaWdodHMKICAgICAgICBlcyBlcXVpdmFsZW50ZSBhIHVuYSBh'
        'dGVuY2nDs24gc3VhdmUuCiAgICAgICAgQU1CQVMgYXByZW5kZW4gYSBwb25kZXJhciBtZW5zYWpl'
        'cyBwb3IgaW1wb3J0YW5jaWEg4oCUIGxhIGRpZmVyZW5jaWEgZXMgcXVlCiAgICAgICAgc2lnbW9p'
        'ZCBwdWVkZSBzaWxlbmNpYXIgdG9kb3MgbG9zIG1lbnNhamVzIChzY29yZSDihpIgMCkgc2kgc29u'
        'IGlycmVsZXZhbnRlcy4KClNDQVRURVIgUEFUVEVSTjoKICAgIExvcyBtZW5zYWplcyBlc3TDoW4g'
        'aW5kZXhhZG9zIHBvciBhcmlzdGEgKG5vIHBvciBub2RvKToKICAgICAgICBtZXNzYWdlczogICAg'
        'ICAgW0UsIG1lc3NhZ2VfZGltXQogICAgICAgIHRhcmdldF9pbmRpY2VzOiBbRV0gIOKAlCDDrW5k'
        'aWNlIGRlbCBub2RvIGRlc3Rpbm8gZGUgY2FkYSBtZW5zYWplCgogICAgU2NhdHRlciBhZGQ6IHBh'
        'cmEgY2FkYSBub2RvIGksIGFncmVnYSBtZW5zYWplcyBkb25kZSB0YXJnZXRfaW5kaWNlcyA9PSBp'
        'LgogICAgSW1wbGVtZW50YWRvIGNvbiB0b3JjaC5zY2F0dGVyX2FkZF8gKGRpc3BvbmlibGUgZW4g'
        'UHlUb3JjaCDiiaUgMS43KS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25z'
        'CgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpmcm9tIC5jb25maWcgaW1wb3J0'
        'IENSRUNvbmZpZwoKCmNsYXNzIEF0dGVudGl2ZUFnZ3JlZ2F0b3Iobm4uTW9kdWxlKToKICAgICIi'
        'IgogICAgQWdyZWdhIG1lbnNhamVzIGVudHJhbnRlcyBwb25kZXJhZG9zIHBvciBpbXBvcnRhbmNp'
        'YSBhcHJlbmRpZGEuCgogICAgVXNvOgogICAgICAgIGNmZyAgPSBDUkVDb25maWcoKQogICAgICAg'
        'IGFnZyAgPSBBdHRlbnRpdmVBZ2dyZWdhdG9yKGNmZykKICAgICAgICAjIG1lc3NhZ2VzOiBbRSwg'
        'TV0sIHRhcmdldHM6IFtFXSwgbm9kZXM6IFtOLCBEXQogICAgICAgIG91dCAgPSBhZ2cobWVzc2Fn'
        'ZXMsIHRhcmdldF9pbmRpY2VzLCBub2RlX2ZlYXR1cmVzLCBuX25vZGVzPU4pCiAgICAgICAgIyBv'
        'dXQ6IFtOLCBNXQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogQ1JFQ29u'
        'ZmlnKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIE0gPSBjb25m'
        'aWcubWVzc2FnZV9kaW0KICAgICAgICBEID0gY29uZmlnLm5vZGVfZGltCgogICAgICAgICMgUHVu'
        'dMO6YSBsYSBpbXBvcnRhbmNpYSBkZSBjYWRhIG1lbnNhamUgZGFkbyBlbCBlc3RhZG8gZGVsIG5v'
        'ZG8gZGVzdGluby4KICAgICAgICAjIElucHV0OiBjb25jYXRlbmFjacOzbiBkZSBbbWVuc2FqZSwg'
        'ZXN0YWRvX2Rlc3Rpbm9dIOKGkiB1biBlc2NhbGFyIGVuICgwLDEpCiAgICAgICAgc2VsZi5hdHRu'
        'X3Njb3JlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihNICsgRCwgTSAv'
        'LyAyKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoTSAvLyAy'
        'LCAxLCBiaWFzPUZhbHNlKSwKICAgICAgICApCgogICAgICAgICMgTm9ybWFsaXphY2nDs24gcG9z'
        'dC1hZ3JlZ2FjacOzbiBwYXJhIGVzdGFiaWxpZGFkCiAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5'
        'ZXJOb3JtKE0sIGVwcz1jb25maWcubm9ybV9lcHMpCgogICAgICAgIHNlbGYuX21lc3NhZ2VfZGlt'
        'ID0gTQogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0X3dlaWdodHMo'
        'c2VsZikgLT4gTm9uZToKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAg'
        'ICAgaWYgaXNpbnN0YW5jZShtLCBubi5MaW5lYXIpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5u'
        'b3JtYWxfKG0ud2VpZ2h0LCBzdGQ9MC4wMikKICAgICAgICAgICAgICAgIGlmIG0uYmlhcyBpcyBu'
        'b3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBubi5pbml0Lnplcm9zXyhtLmJpYXMpCgogICAg'
        'ZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICBtZXNzYWdlczogICAgICAgdG9yY2gu'
        'VGVuc29yLCAgIyBbRSwgbWVzc2FnZV9kaW1dCiAgICAgICAgdGFyZ2V0X2luZGljZXM6IHRvcmNo'
        'LlRlbnNvciwgICMgW0VdICBsb25nIHRlbnNvcgogICAgICAgIG5vZGVfZmVhdHVyZXM6ICB0b3Jj'
        'aC5UZW5zb3IsICAjIFtOLCBub2RlX2RpbV0KICAgICAgICBuX25vZGVzOiAgICAgICAgaW50LAog'
        'ICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAg'
        'ICAgbWVzc2FnZXM6ICAgICAgIFtFLCBNXSDigJQgdW5vIHBvciBhcmlzdGEgZGVsIGdyYWZvCiAg'
        'ICAgICAgICAgIHRhcmdldF9pbmRpY2VzOiBbRV0gICAg4oCUIGEgcXXDqSBub2RvIHZhIGNhZGEg'
        'bWVuc2FqZQogICAgICAgICAgICBub2RlX2ZlYXR1cmVzOiAgW04sIERdIOKAlCBlc3RhZG8gYWN0'
        'dWFsIGRlIGxvcyBub2RvcyAoY29tbyBxdWVyeSkKICAgICAgICAgICAgbl9ub2RlczogICAgICAg'
        'IGludCAgICDigJQgTiAobmVjZXNhcmlvIGN1YW5kbyBFPTApCgogICAgICAgIFJldHVybnM6CiAg'
        'ICAgICAgICAgIGFnZ3JlZ2F0ZWQ6IFtOLCBNXSDigJQgbWVuc2FqZXMgYWdyZWdhZG9zIHBvciBu'
        'b2RvCiAgICAgICAgICAgICAgICAgICAgICAgIHplcm9zIHBhcmEgbm9kb3Mgc2luIG1lbnNhamVz'
        'IGVudHJhbnRlcwogICAgICAgICIiIgogICAgICAgIE0gPSBzZWxmLl9tZXNzYWdlX2RpbQogICAg'
        'ICAgIGRldmljZSA9IG5vZGVfZmVhdHVyZXMuZGV2aWNlCiAgICAgICAgZHR5cGUgID0gbm9kZV9m'
        'ZWF0dXJlcy5kdHlwZQoKICAgICAgICAjIOKUgOKUgCBDYXNvIHNpbiBhcmlzdGFzIOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIGlmIG1lc3NhZ2VzLnNoYXBlWzBd'
        'ID09IDA6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC56ZXJvcyhuX25vZGVzLCBNLCBkZXZpY2U9'
        'ZGV2aWNlLCBkdHlwZT1kdHlwZSkKCiAgICAgICAgRSA9IG1lc3NhZ2VzLnNoYXBlWzBdCgogICAg'
        'ICAgICMg4pSA4pSAIDEuIEF0ZW5jacOzbjogcHVudMO6YSBjYWRhIG1lbnNhamUgdnMuIGVzdGFk'
        'byBkZWwgbm9kbyBkZXN0aW5vIOKUgOKUgOKUgOKUgOKUgAogICAgICAgICMgdGFyZ2V0X2ZlYXR1'
        'cmVzOiBbRSwgRF0g4oCUIGVzdGFkbyBkZWwgbm9kbyBxdWUgUkVDSUJJUsOBIGNhZGEgbWVuc2Fq'
        'ZQogICAgICAgIHRhcmdldF9mZWF0cyA9IG5vZGVfZmVhdHVyZXNbdGFyZ2V0X2luZGljZXNdICAg'
        'ICAgICAgICMgW0UsIERdCiAgICAgICAgYXR0bl9pbnB1dCAgID0gdG9yY2guY2F0KFttZXNzYWdl'
        'cywgdGFyZ2V0X2ZlYXRzXSwgZGltPS0xKSAgIyBbRSwgTStEXQogICAgICAgIHJhd19zY29yZXMg'
        'ICA9IHNlbGYuYXR0bl9zY29yZXIoYXR0bl9pbnB1dCkuc3F1ZWV6ZSgtMSkgICAgICMgW0VdCiAg'
        'ICAgICAgd2VpZ2h0cyAgICAgID0gdG9yY2guc2lnbW9pZChyYXdfc2NvcmVzKSAgICAgICAgICAg'
        'ICAgICAgICAgIyBbRV0g4oiIICgwLDEpCgogICAgICAgICMg4pSA4pSAIDIuIE1lbnNhamVzIHBv'
        'bmRlcmFkb3Mg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgd2VpZ2h0ZWQgPSBtZXNzYWdlcyAq'
        'IHdlaWdodHMudW5zcXVlZXplKC0xKSAgICAgICAgICAgIyBbRSwgTV0KCiAgICAgICAgIyDilIDi'
        'lIAgMy4gQWN1bXVsYXIgbWVuc2FqZXMgcG9yIG5vZG8gZGVzdGlubyDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIAKICAgICAgICAjIGFnZ3JlZ2F0ZWRbaV0gPSDOoyB3ZWlnaHRlZFtlXSBwYXJh'
        'IHRvZG8gZSBjb24gdGFyZ2V0X2luZGljZXNbZV0gPT0gaQogICAgICAgICMKICAgICAgICAjIFVz'
        'YW1vcyBpbmRleF9hZGRfIGVuIGx1Z2FyIGRlIHNjYXR0ZXJfYWRkXyBwb3JxdWUgc2NhdHRlcl9h'
        'ZGRfIDJECiAgICAgICAgIyBubyBlc3TDoSBzb3BvcnRhZG8gZW4gdG9yY2gtZGlyZWN0bWwgKGJh'
        'Y2tlbmQgRGlyZWN0TUwgLyBBTUQgLyBJbnRlbCkuCiAgICAgICAgIyBpbmRleF9hZGRfIGVzIGVx'
        'dWl2YWxlbnRlIHkgZGlmZXJlbmNpYWJsZSBlbiBDUFUsIENVREEgeSBETUwuCiAgICAgICAgIwog'
        'ICAgICAgICMgVXNlIHdlaWdodGVkLmR0eXBlIHNvIGluZGV4X2FkZF8gZHR5cGUgbWF0Y2hlcyB1'
        'bmRlciBBTVAgKEZQMTYvRlAzMikuCiAgICAgICAgYWdncmVnYXRlZCA9IHRvcmNoLnplcm9zKG5f'
        'bm9kZXMsIE0sIGRldmljZT1kZXZpY2UsIGR0eXBlPXdlaWdodGVkLmR0eXBlKQogICAgICAgIGFn'
        'Z3JlZ2F0ZWQuaW5kZXhfYWRkXygwLCB0YXJnZXRfaW5kaWNlcywgd2VpZ2h0ZWQpCgogICAgICAg'
        'ICMg4pSA4pSAIDQuIE5vcm1hbGl6YXIgcG9yIHN1bWEgZGUgcGVzb3MgKGVuIGx1Z2FyIGRlIGNv'
        'dW50KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICAj'
        'IHN1bV93ZWlnaHRzW2ldID0gzqMgd2VpZ2h0c1tlXSBwYXJhIHRvZG8gZSBjb24gdGFyZ2V0X2lu'
        'ZGljZXNbZV0gPT0gaQogICAgICAgIHN1bV93ZWlnaHRzID0gdG9yY2guemVyb3Mobl9ub2Rlcywg'
        'ZGV2aWNlPWRldmljZSwgZHR5cGU9d2VpZ2h0cy5kdHlwZSkKICAgICAgICBzdW1fd2VpZ2h0cy5p'
        'bmRleF9hZGRfKDAsIHRhcmdldF9pbmRpY2VzLCB3ZWlnaHRzKQogICAgICAgIGFnZ3JlZ2F0ZWQg'
        'ICA9IGFnZ3JlZ2F0ZWQgLyAoc3VtX3dlaWdodHMudW5zcXVlZXplKC0xKSArIDFlLTgpCgogICAg'
        'ICAgICMg4pSA4pSAIDUuIExheWVyTm9ybSBwYXJhIGVzdGFiaWxpZGFkIG51bcOpcmljYSDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIAKICAgICAgICByZXR1cm4gc2VsZi5ub3JtKGFnZ3JlZ2F0ZWQp'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtOLCBNXQo='
    ),
    'cre/message_passing.py': (
        '...'
    ),
    'cre/scratch_pad.py': (
        '...'
        'dAo='
    ),
    'cre/engine.py': (
        '...'
    ),
    'decoder/__init__.py': (
        'IiIiCkFJT04tQyBkZWNvZGVyIOKAlCBTdHJlYW1EZWNvZGVyLgoKRGVjb2RpZmljYWRvciBhdXRv'
        'cmVncmVzaXZvIGNvbmRpY2lvbmFkbyBlbiBlbCBncmFmbyBjYXVzYWwgZGVsIENSRS4KQ2F1c2Fs'
        'R3JhcGgg4oaSIG5vZGVfZmVhdHVyZXMgKyB0b2tlbl9pZHMg4oaSIHRva2VuIGxvZ2l0cy4KIiIi'
        'Cgpmcm9tIC5jb25maWcgaW1wb3J0IFN0cmVhbURlY29kZXJDb25maWcKZnJvbSAuaHlicmlkX2xh'
        'eWVyIGltcG9ydCBIeWJyaWREZWNvZGVyTGF5ZXIKZnJvbSAubWV0YV9oZWFkIGltcG9ydCBNZXRh'
        'T3V0cHV0LCBPdXRwdXRNZXRhSGVhZApmcm9tIC5tb2RlbCBpbXBvcnQgRGVjb2Rlck91dHB1dCwg'
        'U3RyZWFtRGVjb2RlcgoKX19hbGxfXyA9IFsKICAgICJEZWNvZGVyT3V0cHV0IiwKICAgICJIeWJy'
        'aWREZWNvZGVyTGF5ZXIiLAogICAgIk1ldGFPdXRwdXQiLAogICAgIk91dHB1dE1ldGFIZWFkIiwK'
        'ICAgICJTdHJlYW1EZWNvZGVyIiwKICAgICJTdHJlYW1EZWNvZGVyQ29uZmlnIiwKXQo='
    ),
    'decoder/config.py': (
        'IiIiCmRlY29kZXIvY29uZmlnLnB5IOKAlCBTdHJlYW1EZWNvZGVyQ29uZmlnCiIiIgoKZnJvbSBf'
        'X2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKZnJvbSBkYXRhY2xhc3Nl'
        'cyBpbXBvcnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcwpjbGFzcyBTdHJlYW1EZWNvZGVyQ29uZmln'
        'OgogICAgIiIiCiAgICBDb25maWd1cmFjacOzbiBkZWwgU3RyZWFtRGVjb2Rlci4KCiAgICBUaW55'
        'ICh0ZXN0aW5nKToKICAgICAgICBoaWRkZW5fZGltPTI1Niwgbl9sYXllcnM9NCwgdm9jYWJfc2l6'
        'ZT0zMjAwMCwgbl9oZWFkcz00LCBtYXhfZ3JhcGhfbm9kZXM9MzIKCiAgICBub2RlX2RpbSBkZWJl'
        'IGNvaW5jaWRpciBjb24gQ1JFQ29uZmlnLm5vZGVfZGltIChyZXByZXNlbnRhY2lvbmVzIGRlIG5v'
        'ZG9zIGRlbCBncmFmbykuCiAgICBtYXhfc2VxX2xlbiBlcyBlbCBsw61taXRlIGRlIGxhIHRhYmxh'
        'IGRlIHBvc2ljaW9uZXMgYXByZW5kaWRhcy4KICAgICIiIgogICAgdm9jYWJfc2l6ZTogICAgICBp'
        'bnQgICA9IDMyXzAwMAogICAgaGlkZGVuX2RpbTogICAgICBpbnQgICA9IDI1NiAgICAgICMgRDog'
        'ZGltZW5zacOzbiBkZWwgbW9kZWxvCiAgICBuX2xheWVyczogICAgICAgIGludCAgID0gNCAgICAg'
        'ICAgIyBOw7ptZXJvIGRlIEh5YnJpZERlY29kZXJMYXllcnMKICAgIG5faGVhZHM6ICAgICAgICAg'
        'aW50ICAgPSA0ICAgICAgICAjIENhYmV6YXMgZGVsIGNyb3NzLWF0dGVudGlvbgogICAgbm9kZV9k'
        'aW06ICAgICAgICBpbnQgICA9IDI1NiAgICAgICMgRGltZW5zacOzbiBkZSBsb3Mgbm9kZSBmZWF0'
        'dXJlcyBkZWwgZ3JhZm8gKENSRU91dHB1dCkKICAgIG1heF9ncmFwaF9ub2RlczogaW50ICAgPSAz'
        'MiAgICAgICAjIE3DoXhpbW8gZGUgbm9kb3Mg4oCUIGNvaW5jaWRlIGNvbiBDcnlzdGFsbGl6ZXJD'
        'b25maWcubWF4X25vZGVzCiAgICBtYXhfc2VxX2xlbjogICAgIGludCAgID0gMjA0OCAgICAgIyBM'
        'w61taXRlIGRlIGxhIHRhYmxhIGRlIHBvc2ljaW9uZXMgYXByZW5kaWRhcwogICAgIyDilIDilIAg'
        'UGFyw6FtZXRyb3MgTWFtYmEgKGNvbXBhcnRpZG9zIGNvbiBlbCBlbmNvZGVyIFNTTSBpbnRlcm5v'
        'KSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0YXRlX2RpbTogICAg'
        'ICAgaW50ICAgPSAxNiAgICAgICAjIE46IGRpbWVuc2nDs24gZGVsIGVzdGFkbyBTU00KICAgIGV4'
        'cGFuZDogICAgICAgICAgaW50ICAgPSAyICAgICAgICAjIERfaW5uZXIgPSBleHBhbmQgw5cgRAog'
        'ICAgZF9jb252OiAgICAgICAgICBpbnQgICA9IDQgICAgICAgICMgQW5jaG8gZGUgbGEgY29udm9s'
        'dWNpw7NuIGNhdXNhbAogICAgZmZuX211bHQ6ICAgICAgICBpbnQgICA9IDQgICAgICAgICMgR2F0'
        'ZWRGRk4gaW5uZXIgPSBmZm5fbXVsdCDDlyBECiAgICBkcm9wb3V0OiAgICAgICAgIGZsb2F0ID0g'
        'MC4wCiAgICBiaWFzOiAgICAgICAgICAgIGJvb2wgID0gRmFsc2UKICAgIHJtc19lcHM6ICAgICAg'
        'ICAgZmxvYXQgPSAxZS01CiAgICBub3JtX2VwczogICAgICAgIGZsb2F0ID0gMWUtNQoKICAgIGRl'
        'ZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5oaWRkZW5fZGlt'
        'ICUgc2VsZi5uX2hlYWRzICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAg'
        'ICAgICAgICAgICBmImhpZGRlbl9kaW0gKHtzZWxmLmhpZGRlbl9kaW19KSBtdXN0IGJlIGRpdmlz'
        'aWJsZSBieSBuX2hlYWRzICh7c2VsZi5uX2hlYWRzfSkiCiAgICAgICAgICAgICkKICAgICAgICBp'
        'ZiBzZWxmLnZvY2FiX3NpemUgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidm9j'
        'YWJfc2l6ZSBtdXN0IGJlID49IDEsIGdvdCB7c2VsZi52b2NhYl9zaXplfSIpCiAgICAgICAgaWYg'
        'c2VsZi5tYXhfZ3JhcGhfbm9kZXMgPCAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYi'
        'bWF4X2dyYXBoX25vZGVzIG11c3QgYmUgPj0gMSwgZ290IHtzZWxmLm1heF9ncmFwaF9ub2Rlc30i'
        'KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGRfaW5uZXIoc2VsZikgLT4gaW50OgogICAgICAgICIi'
        'IkRpbWVuc2nDs24gaW50ZXJuYSBkZWwgU1NNIChEX2lubmVyID0gZXhwYW5kIMOXIEQpLiIiIgog'
        'ICAgICAgIHJldHVybiBzZWxmLmV4cGFuZCAqIHNlbGYuaGlkZGVuX2RpbQoKICAgIEBwcm9wZXJ0'
        'eQogICAgZGVmIGR0X3Jhbmsoc2VsZikgLT4gaW50OgogICAgICAgICIiIlJhbmdvIGRlIM6UIOKA'
        'lCBpZ3VhbCBxdWUgZW4gZWwgZW5jb2Rlci4iIiIKICAgICAgICByZXR1cm4gbWF0aC5jZWlsKHNl'
        'bGYuaGlkZGVuX2RpbSAvIDE2KQo='
    ),
    'decoder/meta_head.py': (
        'IiIiCmRlY29kZXIvbWV0YV9oZWFkLnB5IOKAlCBPdXRwdXRNZXRhSGVhZAo9PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PQoKUHJlZGljZSBtZXRhZGF0b3MgZGVsIG91dHB1dCBk'
        'ZWwgU3RyZWFtRGVjb2RlcjoKICAgIC0gY29uZmlkZW5jZTogICAgICAgICAgIMK/cXXDqSB0YW4g'
        'c2VndXJvIGVzdMOhIGVsIG1vZGVsbyBkZSBzdSByZXNwdWVzdGE/CiAgICAtIG5lZWRzX2NsYXJp'
        'ZmljYXRpb246ICDCv25lY2VzaXRhIHBlZGlyIG3DoXMgY29udGV4dG8gYWwgdXN1YXJpbz8KCkFS'
        'UVVJVEVDVFVSQToKICAgIENvbWJpbmE6CiAgICAgICAgMS4gUmVwcmVzZW50YWNpw7NuIG1lZGlh'
        'IGRlIGxvcyB0b2tlbnMgZ2VuZXJhZG9zICAobWVhbi1wb29sIHNvYnJlIEwpCiAgICAgICAgMi4g'
        'UmVwcmVzZW50YWNpw7NuIG1lZGlhIGRlbCBncmFmbyBjYXVzYWwgICAgICAgICAobWVhbi1wb29s'
        'IHNvYnJlIG5fbm9kZXMpCgogICAgTHVlZ28gYXBsaWNhIGRvcyBoZWFkcyBpbmRlcGVuZGllbnRl'
        'cyBjb24gc2lnbW9pZCBwYXJhIGVzY2FsYXIgYSBbMCwxXS4KCiAgICBJbnB1dDoKICAgICAgICBo'
        'aWRkZW46ICAgICBbQiwgTCwgaGlkZGVuX2RpbV0gICDigJQgaGlkZGVuIHN0YXRlcyBkZWwgZGVj'
        'b2RlcgogICAgICAgIGdyYXBoX3JlcHI6IFtCLCBuX25vZGVzLCBub2RlX2RpbV0g4oCUIGZlYXR1'
        'cmVzIGRlbCBncmFmbyAoZGVsIENSRSkKCiAgICBPdXRwdXQ6CiAgICAgICAgTWV0YU91dHB1dCBj'
        'b24gY29uZmlkZW5jZSBbQl0geSBuZWVkc19jbGFyaWZpY2F0aW9uIFtCXQoKTk9UQToKICAgIE9w'
        'ZXJhciBzb2JyZSBsYSBtZWRpYSBkZSB0b2tlbnMgKG5vIHNvbG8gZWwgw7psdGltbykgZXMgbcOh'
        'cyBlc3RhYmxlCiAgICBkdXJhbnRlIGVsIGVudHJlbmFtaWVudG8geSBjYXB0dXJhIGxhIGludGVu'
        'Y2nDs24gZ2xvYmFsIGRlbCB0ZXh0byBnZW5lcmFkby4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1w'
        'b3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9y'
        'dCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwg'
        'YXMgRgoKZnJvbSAuY29uZmlnIGltcG9ydCBTdHJlYW1EZWNvZGVyQ29uZmlnCgoKIyDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBP'
        'VVRQVVQgQ09OVEFJTkVSCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpAZGF0YWNsYXNzCmNsYXNzIE1ldGFPdXRwdXQ6CiAgICAi'
        'IiIKICAgIFJlc3VsdGFkbyBkZWwgT3V0cHV0TWV0YUhlYWQuCgogICAgY29uZmlkZW5jZTogICAg'
        'ICAgICAgIFtCXSDiiIggWzAsIDFdIOKAlCBjb25maWFuemEgZW4gZWwgb3V0cHV0IGdlbmVyYWRv'
        'CiAgICBuZWVkc19jbGFyaWZpY2F0aW9uOiAgW0JdIOKIiCBbMCwgMV0g4oCUIHByb2JhYmlsaWRh'
        'ZCBkZSBuZWNlc2l0YXIgY2xhcmlmaWNhY2nDs24KICAgICIiIgogICAgY29uZmlkZW5jZTogICAg'
        'ICAgICAgdG9yY2guVGVuc29yCiAgICBuZWVkc19jbGFyaWZpY2F0aW9uOiB0b3JjaC5UZW5zb3IK'
        'CgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgAojIE1FVEEgSEVBRAojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgT3V0cHV0TWV0YUhlYWQobm4uTW9kdWxl'
        'KToKICAgICIiIgogICAgUHJlZGljZSBjb25maWRlbmNlIHkgbmVlZHNfY2xhcmlmaWNhdGlvbiBk'
        'ZWwgb3V0cHV0IGdlbmVyYWRvLgoKICAgIFVzbzoKICAgICAgICBtZXRhX2hlYWQgPSBPdXRwdXRN'
        'ZXRhSGVhZChjb25maWcpCiAgICAgICAgbWV0YSA9IG1ldGFfaGVhZChoaWRkZW4sIGdyYXBoX3Jl'
        'cHIpCiAgICAgICAgIyBtZXRhLmNvbmZpZGVuY2U6ICAgICAgICAgIFtCXQogICAgICAgICMgbWV0'
        'YS5uZWVkc19jbGFyaWZpY2F0aW9uOiBbQl0KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxm'
        'LCBjb25maWc6IFN0cmVhbURlY29kZXJDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5f'
        'X2luaXRfXygpCiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGltCiAgICAgICAgRyA9IGNvbmZp'
        'Zy5ub2RlX2RpbQoKICAgICAgICAjIFByb3llY2Npw7NuIGNvbWJpbmFkYTogKHRva2VuX3Bvb2wg'
        '4oCWIGdyYXBoX3Bvb2wpIOKGkiBoaWRkZW4KICAgICAgICBzZWxmLnByb2ogPSBubi5MaW5lYXIo'
        'RCArIEcsIEQpCgogICAgICAgICMgRG9zIGhlYWRzIGluZGVwZW5kaWVudGVzIOKGkiBlc2NhbGFy'
        'ZXMgcG9yIGVqZW1wbG8KICAgICAgICBzZWxmLmNvbmZpZGVuY2VfaGVhZCAgICA9IG5uLkxpbmVh'
        'cihELCAxKQogICAgICAgIHNlbGYuY2xhcmlmaWNhdGlvbl9oZWFkID0gbm4uTGluZWFyKEQsIDEp'
        'CgogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIF9pbml0X3dlaWdodHMoc2Vs'
        'ZikgLT4gTm9uZToKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5wcm9qLndlaWdodCwgc3Rk'
        'PTAuMDIpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5wcm9qLmJpYXMpCiAgICAgICAgbm4u'
        'aW5pdC5ub3JtYWxfKHNlbGYuY29uZmlkZW5jZV9oZWFkLndlaWdodCwgc3RkPTAuMDIpCiAgICAg'
        'ICAgbm4uaW5pdC56ZXJvc18oc2VsZi5jb25maWRlbmNlX2hlYWQuYmlhcykKICAgICAgICBubi5p'
        'bml0Lm5vcm1hbF8oc2VsZi5jbGFyaWZpY2F0aW9uX2hlYWQud2VpZ2h0LCBzdGQ9MC4wMikKICAg'
        'ICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmNsYXJpZmljYXRpb25faGVhZC5iaWFzKQoKICAgIGRl'
        'ZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgaGlkZGVuOiAgICAgdG9yY2guVGVuc29y'
        'LCAgICMgW0IsIEwsIGhpZGRlbl9kaW1dCiAgICAgICAgZ3JhcGhfcmVwcjogdG9yY2guVGVuc29y'
        'LCAgICMgW0IsIG5fbm9kZXMsIG5vZGVfZGltXQogICAgKSAtPiBNZXRhT3V0cHV0OgogICAgICAg'
        'ICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGhpZGRlbjogICAgIFtCLCBMLCBoaWRkZW5f'
        'ZGltXSAgICDigJQgaGlkZGVuIHN0YXRlcyBmaW5hbGVzIGRlbCBkZWNvZGVyCiAgICAgICAgICAg'
        'IGdyYXBoX3JlcHI6IFtCLCBuX25vZGVzLCBub2RlX2RpbV0g4oCUIGZlYXR1cmVzIGRlbCBncmFm'
        'byBjYXVzYWwKCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgTWV0YU91dHB1dCBjb24gdGVu'
        'c29yZXMgW0JdIOKIiCBbMCwgMV0KICAgICAgICAiIiIKICAgICAgICAjIE1lYW4tcG9vbCBzb2Jy'
        'ZSBsYSBkaW1lbnNpw7NuIGRlIHNlY3VlbmNpYSAvIG5vZG9zCiAgICAgICAgaF9wb29sID0gaGlk'
        'ZGVuLm1lYW4oZGltPTEpICAgICAgIyBbQiwgRF0KICAgICAgICBnX3Bvb2wgPSBncmFwaF9yZXBy'
        'Lm1lYW4oZGltPTEpICAjIFtCLCBHXQoKICAgICAgICBjb21iaW5lZCA9IHRvcmNoLmNhdChbaF9w'
        'b29sLCBnX3Bvb2xdLCBkaW09LTEpICAjIFtCLCBEK0ddCiAgICAgICAgaCA9IEYuZ2VsdShzZWxm'
        'LnByb2ooY29tYmluZWQpKSAgICAgICAgICAgICAgICAgICMgW0IsIERdCgogICAgICAgIGNvbmZp'
        'ZGVuY2UgICAgICAgICAgPSB0b3JjaC5zaWdtb2lkKHNlbGYuY29uZmlkZW5jZV9oZWFkKGgpLnNx'
        'dWVlemUoLTEpKSAgICAgIyBbQl0KICAgICAgICBuZWVkc19jbGFyaWZpY2F0aW9uID0gdG9yY2gu'
        'c2lnbW9pZChzZWxmLmNsYXJpZmljYXRpb25faGVhZChoKS5zcXVlZXplKC0xKSkgICMgW0JdCgog'
        'ICAgICAgIHJldHVybiBNZXRhT3V0cHV0KAogICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVu'
        'Y2UsCiAgICAgICAgICAgIG5lZWRzX2NsYXJpZmljYXRpb249bmVlZHNfY2xhcmlmaWNhdGlvbiwK'
        'ICAgICAgICApCg=='
    ),
    'decoder/hybrid_layer.py': (
        'IiIiCmRlY29kZXIvaHlicmlkX2xheWVyLnB5IOKAlCBIeWJyaWREZWNvZGVyTGF5ZXIKPT09PT09'
        'PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpDYXBhIGjDrWJyaWRhIGRl'
        'bCBTdHJlYW1EZWNvZGVyIHF1ZSBjb21iaW5hIGN1YXRybyBzdWItbcOzZHVsb3MgZW4gc2VjdWVu'
        'Y2lhOgoKICAgIDEuIE1hbWJhTGF5ZXIgKHJldXRpbGl6YWRvIGRlIGVuY29kZXIvbWFtYmFfbGF5'
        'ZXIucHkpCiAgICAgICBQcm9jZXNhbWllbnRvIENBVVNBTCBkZSBsYSBzZWN1ZW5jaWEgZGUgdG9r'
        'ZW5zLgogICAgICAgTyhMKSBlbiBtZW1vcmlhIHkgY29tcHV0ZSDigJQgaGVyZWRhIHRvZG9zIGxv'
        'cyBiZW5lZmljaW9zIGRlbCBTU00uCgogICAgMi4gQ3Jvc3MtYXR0ZW50aW9uIGFsIGdyYWZvIGNh'
        'dXNhbCAobm4uTXVsdGloZWFkQXR0ZW50aW9uKQogICAgICAgTG9zIHRva2VucyAiY29uc3VsdGFu'
        'IiBlbCBncmFmbyBwYXJhIGFuY2xhciBzdSBzZW3DoW50aWNhIGNhdXNhbC4KICAgICAgIFEgPSB0'
        'b2tlbnMgIFtCLCBMLCBEXQogICAgICAgSyA9IFYgPSBub2RvcyBkZWwgZ3JhZm8gIFtCLCBuX25v'
        'ZGVzLCBEXQoKICAgIDMuIENyb3NzLWF0dGVudGlvbiBhIGxvcyBjb25jZXB0IHZlY3RvcnMgZGVs'
        'IGVuY29kZXIgKG5uLk11bHRpaGVhZEF0dGVudGlvbikKICAgICAgIExvcyB0b2tlbnMgImNvbnN1'
        'bHRhbiIgZGlyZWN0YW1lbnRlIGVsIGlucHV0IHBhcmEgcHJlc2VydmFyIGlkZW50aWRhZCBsw6l4'
        'aWNhLgogICAgICAgUSA9IHRva2VucyAgW0IsIEwsIERdCiAgICAgICBLID0gViA9IGNvbmNlcHQg'
        'dmVjdG9ycyBkZWwgZW5jb2RlciAgW0IsIExfZW5jLCBEXQogICAgICAgUmVzdWVsdmUgZWwgcHJv'
        'YmxlbWEgZGUgZ3JvdW5kaW5nOiBzaW4gZXN0YSBjYXBhLCBsYSBpZGVudGlkYWQgbMOpeGljYQog'
        'ICAgICAgKCJmaWVicmUiIHZzICJpbmNlbmRpbyIpIHNlIHBpZXJkZSBhbCBwYXNhciBwb3IgY3J5'
        'c3RhbGxpemVyIOKGkiBDUkUuCiAgICAgICBFbCB0cmFuc2Zvcm1lciBubyB0aWVuZSBlc3RlIHBy'
        'b2JsZW1hIHBvcnF1ZSBzdSBjcm9zcy1hdHRlbnRpb24gdmUKICAgICAgIGRpcmVjdGFtZW50ZSBs'
        'b3MgdG9rZW5zIGRlbCBlbmNvZGVyIOKAlCBlc3RhIGNhcGEgcmVwbGljYSBlc2EgcHJvcGllZGFk'
        'LgoKICAgIDQuIEdhdGVkRkZOIChyZXV0aWxpemFkbyBkZSBlbmNvZGVyL21hbWJhX2xheWVyLnB5'
        'KQogICAgICAgUHJvY2VzYW1pZW50byBhZGljaW9uYWwgcG9yIHRva2VuIGRlc3B1w6lzIGRlIGlu'
        'dGVncmFyIGFtYm9zIGNvbnRleHRvcy4KCkZMVUpPOgogICAgeCBbQiwgTCwgRF0KICAgIOKGkiBN'
        'YW1iYUxheWVyIChTU00gKyBGRk4gaW50ZXJubykgICAgICAgICAgICAgIOKGkiB4JyAgIFtCLCBM'
        'LCBEXQogICAg4oaSIExOIOKGkiBDcm9zc0F0dG4oZ3JhcGhfcmVwcikgKyByZXNpZCAgICAgICAg'
        'ICDihpIgeCcnICBbQiwgTCwgRF0KICAgIOKGkiBMTiDihpIgQ3Jvc3NBdHRuKGVuY29kZXJfY29u'
        'Y2VwdHMpICsgcmVzaWQgICDihpIgeCcnJyBbQiwgTCwgRF0KICAgIOKGkiBMTiDihpIgR2F0ZWRG'
        'Rk4gKyByZXNpZCAgICAgICAgICAgICAgICAgICAgICAg4oaSIHgnJycnIFtCLCBMLCBEXQoKUkVV'
        'VElMSVpBQ0nDk04gREUgRU5DT0RFUjoKICAgIE1hbWJhTGF5ZXIgeSBHYXRlZEZGTiBzZSBpbXBv'
        'cnRhbiBkZSBlbmNvZGVyLm1hbWJhX2xheWVyIHNpbiBtb2RpZmljYWNpw7NuLgogICAgRWwgSHli'
        'cmlkRGVjb2RlckxheWVyIGNyZWEgdW4gU3RyZWFtRW5jb2RlckNvbmZpZyBjb21wYXRpYmxlIHVz'
        'YW5kbwogICAgbG9zIGhpcGVycGFyw6FtZXRyb3MgZGVsIFN0cmVhbURlY29kZXJDb25maWcuCiIi'
        'IgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwppbXBvcnQg'
        'b3MKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUoX19m'
        'aWxlX18pKSkKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCmZyb20gZW5jb2Rl'
        'ci5tYW1iYV9sYXllciBpbXBvcnQgR2F0ZWRGRk4sIE1hbWJhTGF5ZXIsIFN0cmVhbUVuY29kZXJD'
        'b25maWcKZnJvbSAuY29uZmlnIGltcG9ydCBTdHJlYW1EZWNvZGVyQ29uZmlnCgoKIyDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBI'
        'RUxQRVI6IGNvbnN0cnVpciBTdHJlYW1FbmNvZGVyQ29uZmlnIGRlc2RlIFN0cmVhbURlY29kZXJD'
        'b25maWcKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIAKCmRlZiBfbWFrZV9tYW1iYV9jb25maWcoY2ZnOiBTdHJlYW1EZWNvZGVyQ29u'
        'ZmlnKSAtPiBTdHJlYW1FbmNvZGVyQ29uZmlnOgogICAgIiIiCiAgICBDb25zdHJ1eWUgdW4gU3Ry'
        'ZWFtRW5jb2RlckNvbmZpZyBjb21wYXRpYmxlIHBhcmEgaW5zdGFuY2lhciBNYW1iYUxheWVyLgog'
        'ICAgRWwgTWFtYmFMYXllciBkZWwgZW5jb2RlciBlcyBpZMOpbnRpY28gYWwgcXVlIG5lY2VzaXRh'
        'bW9zIGFxdcOtIOKAlAogICAgbGEgw7puaWNhIGRpZmVyZW5jaWEgZXMgZWwgY29udGV4dG8gKGRl'
        'Y29kZXIgdnMgZW5jb2RlcikuCiAgICAiIiIKICAgIHJldHVybiBTdHJlYW1FbmNvZGVyQ29uZmln'
        'KAogICAgICAgIHZvY2FiX3NpemUgID0gY2ZnLnZvY2FiX3NpemUsCiAgICAgICAgaGlkZGVuX2Rp'
        'bSAgPSBjZmcuaGlkZGVuX2RpbSwKICAgICAgICBuX2xheWVycyAgICA9IDEsICAgICAgICAgICAg'
        'ICMgc29sbyB1c2Ftb3MgMSBsYXllciBhIGxhIHZlegogICAgICAgIHN0YXRlX2RpbSAgID0gY2Zn'
        'LnN0YXRlX2RpbSwKICAgICAgICBleHBhbmQgICAgICA9IGNmZy5leHBhbmQsCiAgICAgICAgZF9j'
        'b252ICAgICAgPSBjZmcuZF9jb252LAogICAgICAgIGZmbl9tdWx0ICAgID0gY2ZnLmZmbl9tdWx0'
        'LAogICAgICAgIGRyb3BvdXQgICAgID0gY2ZnLmRyb3BvdXQsCiAgICAgICAgYmlhcyAgICAgICAg'
        'PSBjZmcuYmlhcywKICAgICAgICBybXNfZXBzICAgICA9IGNmZy5ybXNfZXBzLAogICAgKQoKCiMg'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSACiMgSFlCUklEIERFQ09ERVIgTEFZRVIKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIEh5YnJpZERlY29kZXJMYXll'
        'cihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBNYW1iYUxheWVyICsgY3Jvc3MtYXR0ZW50aW9uIGFs'
        'IGdyYWZvICsgR2F0ZWRGRk4uCgogICAgVXNvOgogICAgICAgIGxheWVyID0gSHlicmlkRGVjb2Rl'
        'ckxheWVyKGNvbmZpZykKICAgICAgICB4X291dCA9IGxheWVyKHgsIGdyYXBoX3JlcHIpCiAgICAg'
        'ICAgIyB4X291dDogW0IsIEwsIGhpZGRlbl9kaW1dCgogICAgQXJnczoKICAgICAgICBjb25maWc6'
        'IFN0cmVhbURlY29kZXJDb25maWcKICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25m'
        'aWc6IFN0cmVhbURlY29kZXJDb25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRf'
        'XygpCiAgICAgICAgRCA9IGNvbmZpZy5oaWRkZW5fZGltCiAgICAgICAgRyA9IGNvbmZpZy5ub2Rl'
        'X2RpbQoKICAgICAgICAjIOKUgOKUgCAxLiBNYW1iYSBTU00gYmxvY2sgKGNvbiBzdSBHYXRlZEZG'
        'TiBpbnRlcm5vKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIAKICAgICAgICBtYW1iYV9jZmcgPSBfbWFrZV9tYW1iYV9jb25m'
        'aWcoY29uZmlnKQogICAgICAgIHNlbGYubWFtYmEgPSBNYW1iYUxheWVyKG1hbWJhX2NmZykKCiAg'
        'ICAgICAgIyDilIDilIAgMi4gQ3Jvc3MtYXR0ZW50aW9uIGhhY2lhIGVsIGdyYWZvIChlc3RydWN0'
        'dXJhIGNhdXNhbCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAg'
        'ICAgc2VsZi5jcm9zc19hdHRuX25vcm0gPSBubi5MYXllck5vcm0oRCwgZXBzPWNvbmZpZy5ub3Jt'
        'X2VwcykKCiAgICAgICAgc2VsZi5jcm9zc19hdHRuID0gbm4uTXVsdGloZWFkQXR0ZW50aW9uKAog'
        'ICAgICAgICAgICBlbWJlZF9kaW0gICA9IEQsCiAgICAgICAgICAgIG51bV9oZWFkcyAgID0gY29u'
        'ZmlnLm5faGVhZHMsCiAgICAgICAgICAgIGJhdGNoX2ZpcnN0ID0gVHJ1ZSwKICAgICAgICAgICAg'
        'YmlhcyAgICAgICAgPSBjb25maWcuYmlhcywKICAgICAgICAgICAgZHJvcG91dCAgICAgPSBjb25m'
        'aWcuZHJvcG91dCwKICAgICAgICApCgogICAgICAgICMgUHJveWVjdGEgbm9kZSBmZWF0dXJlcyBh'
        'bCBlc3BhY2lvIGRlbCBkZWNvZGVyIHNpIGRpZmllcmVuIGVuIGRpbWVuc2nDs24KICAgICAgICBz'
        'ZWxmLmdyYXBoX3Byb2o6IG5uLk1vZHVsZSA9ICgKICAgICAgICAgICAgbm4uTGluZWFyKEcsIEQs'
        'IGJpYXM9RmFsc2UpIGlmIEcgIT0gRCBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICApCgogICAg'
        'ICAgICMg4pSA4pSAIDMuIENyb3NzLWF0dGVudGlvbiBhIGNvbmNlcHRzIGRlbCBlbmNvZGVyIChp'
        'ZGVudGlkYWQgbMOpeGljYSkg4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgIyBQZXJtaXRl'
        'IHF1ZSBjYWRhIHRva2VuIGdlbmVyYWRvICJsZWEiIGRpcmVjdGFtZW50ZSBsb3MgY29uY2VwdCB2'
        'ZWN0b3JzCiAgICAgICAgIyBkZWwgZW5jb2RlciwgcHJlc2VydmFuZG8gbGEgaWRlbnRpZGFkIGzD'
        'qXhpY2Egb3JpZ2luYWwgcXVlIGVsCiAgICAgICAgIyBjcnlzdGFsbGl6ZXIg4oaSIENSRSBwdWVk'
        'ZSBhYnN0cmFlciBvIHRyYW5zZm9ybWFyLgogICAgICAgIHNlbGYuZW5jX2F0dG5fbm9ybSA9IG5u'
        'LkxheWVyTm9ybShELCBlcHM9Y29uZmlnLm5vcm1fZXBzKQoKICAgICAgICBzZWxmLmVuY19hdHRu'
        'ID0gbm4uTXVsdGloZWFkQXR0ZW50aW9uKAogICAgICAgICAgICBlbWJlZF9kaW0gICA9IEQsCiAg'
        'ICAgICAgICAgIG51bV9oZWFkcyAgID0gY29uZmlnLm5faGVhZHMsCiAgICAgICAgICAgIGJhdGNo'
        'X2ZpcnN0ID0gVHJ1ZSwKICAgICAgICAgICAgYmlhcyAgICAgICAgPSBjb25maWcuYmlhcywKICAg'
        'ICAgICAgICAgZHJvcG91dCAgICAgPSBjb25maWcuZHJvcG91dCwKICAgICAgICApCgogICAgICAg'
        'ICMg4pSA4pSAIDQuIEdhdGVkRkZOIGFkaWNpb25hbCAocG9zdC1jcm9zcy1hdHRlbnRpb25zKSDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIAKICAgICAgICBzZWxmLmZmbl9ub3JtID0gbm4uTGF5ZXJOb3JtKEQsIGVwcz1jb25maWcu'
        'bm9ybV9lcHMpCiAgICAgICAgc2VsZi5mZm4gICAgICA9IEdhdGVkRkZOKEQsIGZmbl9tdWx0PWNv'
        'bmZpZy5mZm5fbXVsdCwgYmlhcz1jb25maWcuYmlhcykKCiAgICAgICAgc2VsZi5kcm9wID0gKAog'
        'ICAgICAgICAgICBubi5Ecm9wb3V0KGNvbmZpZy5kcm9wb3V0KSBpZiBjb25maWcuZHJvcG91dCA+'
        'IDAuMCBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICApCgogICAgICAgIHNlbGYuX2luaXRfd2Vp'
        'Z2h0cygpCgogICAgZGVmIF9pbml0X3dlaWdodHMoc2VsZikgLT4gTm9uZToKICAgICAgICAjIGdy'
        'YXBoX3Byb2ogcHVlZGUgc2VyIElkZW50aXR5IG8gTGluZWFyCiAgICAgICAgaWYgaXNpbnN0YW5j'
        'ZShzZWxmLmdyYXBoX3Byb2osIG5uLkxpbmVhcik6CiAgICAgICAgICAgIG5uLmluaXQubm9ybWFs'
        'XyhzZWxmLmdyYXBoX3Byb2oud2VpZ2h0LCBzdGQ9MC4wMikKCiAgICBkZWYgZm9yd2FyZCgKICAg'
        'ICAgICBzZWxmLAogICAgICAgIHg6ICAgICAgICAgICAgICAgIHRvcmNoLlRlbnNvciwgICAgICAg'
        'ICAgICAgICMgW0IsIEwsIERdCiAgICAgICAgZ3JhcGhfcmVwcjogICAgICAgdG9yY2guVGVuc29y'
        'LCAgICAgICAgICAgICAgIyBbQiwgbl9ub2RlcywgR10KICAgICAgICBlbmNvZGVyX2NvbmNlcHRz'
        'OiB0b3JjaC5UZW5zb3IgfCBOb25lID0gTm9uZSwgICMgW0IsIExfZW5jLCBEXQogICAgKSAtPiB0'
        'b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgeDogICAg'
        'ICAgICAgICAgICAgW0IsIEwsIGhpZGRlbl9kaW1dICAgICDigJQgaGlkZGVuIHN0YXRlcyBkZSBs'
        'b3MgdG9rZW5zCiAgICAgICAgICAgIGdyYXBoX3JlcHI6ICAgICAgIFtCLCBuX25vZGVzLCBub2Rl'
        'X2RpbV0g4oCUIGZlYXR1cmVzIGRlIGxvcyBub2RvcyBkZWwgZ3JhZm8KICAgICAgICAgICAgZW5j'
        'b2Rlcl9jb25jZXB0czogW0IsIExfZW5jLCBoaWRkZW5fZGltXSDigJQgY29uY2VwdCB2ZWN0b3Jz'
        'IGRlbCBTdHJlYW1FbmNvZGVyCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChvcHRpb25h'
        'bDsgd2hlbiBOb25lLCB0aGUgZW5jb2RlciBjcm9zcy1hdHRlbnRpb24gaXMgc2tpcHBlZCkKCiAg'
        'ICAgICAgUmV0dXJuczoKICAgICAgICAgICAgW0IsIEwsIGhpZGRlbl9kaW1dCiAgICAgICAgIiIi'
        'CiAgICAgICAgIyDilIDilIAgMS4gTWFtYmFMYXllciAoU1NNIGNhdXNhbCArIEZGTiBpbnRlcm5v'
        'LCBwcmUtbm9ybSBpbnRlcm5hbWVudGUpIOKUgOKUgOKUgAogICAgICAgIHgsIF8gPSBzZWxmLm1h'
        'bWJhKHgpICAgIyBbQiwgTCwgRF0KCiAgICAgICAgIyDilIDilIAgMi4gQ3Jvc3MtYXR0ZW50aW9u'
        'IGFsIGdyYWZvIChlc3RydWN0dXJhIGNhdXNhbCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgcmVzaWR1YWwgPSB4CiAgICAg'
        'ICAgaCA9IHNlbGYuY3Jvc3NfYXR0bl9ub3JtKHgpICAgICAgICAgICAgICAgICAgICAjIFtCLCBM'
        'LCBEXQogICAgICAgIGt2ID0gc2VsZi5ncmFwaF9wcm9qKGdyYXBoX3JlcHIpICAgICAgICAgICAg'
        'ICAgIyBbQiwgbl9ub2RlcywgRF0KICAgICAgICBhdHRuX291dCwgXyA9IHNlbGYuY3Jvc3NfYXR0'
        'bigKICAgICAgICAgICAgcXVlcnkgPSBoLAogICAgICAgICAgICBrZXkgICA9IGt2LAogICAgICAg'
        'ICAgICB2YWx1ZSA9IGt2LAogICAgICAgICkgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgIyBbQiwgTCwgRF0KICAgICAgICB4ID0gcmVzaWR1YWwgKyBzZWxmLmRy'
        'b3AoYXR0bl9vdXQpCgogICAgICAgICMg4pSA4pSAIDMuIENyb3NzLWF0dGVudGlvbiBhbCBlbmNv'
        'ZGVyIChpZGVudGlkYWQgbMOpeGljYSkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgaWYgZW5jb2Rlcl9jb25jZXB0cyBpcyBub3Qg'
        'Tm9uZToKICAgICAgICAgICAgcmVzaWR1YWwgPSB4CiAgICAgICAgICAgIGggPSBzZWxmLmVuY19h'
        'dHRuX25vcm0oeCkgICAgICAgICAgICAgICAgICAgICAgIyBbQiwgTCwgRF0KICAgICAgICAgICAg'
        'ZW5jX291dCwgXyA9IHNlbGYuZW5jX2F0dG4oCiAgICAgICAgICAgICAgICBxdWVyeSA9IGgsCiAg'
        'ICAgICAgICAgICAgICBrZXkgICA9IGVuY29kZXJfY29uY2VwdHMsCiAgICAgICAgICAgICAgICB2'
        'YWx1ZSA9IGVuY29kZXJfY29uY2VwdHMsCiAgICAgICAgICAgICkgICAgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbQiwgTCwgRF0KICAgICAgICAgICAgeCA9IHJl'
        'c2lkdWFsICsgc2VsZi5kcm9wKGVuY19vdXQpCgogICAgICAgICMg4pSA4pSAIDQuIEdhdGVkRkZO'
        'IChwcmUtbm9ybSArIHJlc2lkdWFsKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIAKICAgICAgICByZXNpZHVhbCA9IHgKICAgICAgICB4ID0gcmVzaWR1YWwgKyBzZWxm'
        'LmRyb3Aoc2VsZi5mZm4oc2VsZi5mZm5fbm9ybSh4KSkpCgogICAgICAgIHJldHVybiB4Cg=='
    ),
    'decoder/model.py': (
        'IiIiCmRlY29kZXIvbW9kZWwucHkg4oCUIFN0cmVhbURlY29kZXIKPT09PT09PT09PT09PT09PT09'
        'PT09PT09PT09PT09PT09PQoKRWwgZGVjb2RpZmljYWRvciBhdXRvcmVncmVzaXZvIGRlIEFJT04t'
        'QyBjb25kaWNpb25hZG8gZW4gZWwgZ3JhZm8gY2F1c2FsLgoKQVJRVUlURUNUVVJBOgogICAgdG9r'
        'ZW5faWRzIFtCLCBMXQogICAgICAgIOKGkyB0b2tlbl9lbWJlZGRpbmcgKyBwb3NfZW1iZWRkaW5n'
        'CiAgICB4IFtCLCBMLCBEXQogICAgICAgIOKGkyBIeWJyaWREZWNvZGVyTGF5ZXIgw5cgbl9sYXll'
        'cnMKICAgICAgICAgIChNYW1iYUxheWVyICsgY3Jvc3MtYXR0biDihpIgZ3JhZm8gKyBHYXRlZEZG'
        'TikKICAgIHggW0IsIEwsIERdCiAgICAgICAg4oaTIGZpbmFsX25vcm0KICAgIOKUjOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUkAogICAg4pSCICBsbV9oZWFkIFtCLCBMLCB2b2NhYl9zaXplXSAgIOKG'
        'kiB0b2tlbiBsb2dpdHPilIIKICAgIOKUgiAgYW5jaG9yX2hlYWQgW0IsIEwsIG1heF9ub2Rlc13i'
        'hpIgYW5jaG9yIGxvZ2l0c+KUggogICAg4pSCICBtZXRhX2hlYWQg4oaSIGNvbmZpZGVuY2UsIG5l'
        'ZWRzX2NsYXJpZmljYXRpb24g4pSCCiAgICDilJTilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilJgKCkNP'
        'TkRJVElPTklORyBFTiBFTCBHUkFGTyArIEVOQ09ERVI6CiAgICBncmFwaF9yZXByICAgICAgIFtC'
        'LCBuX25vZGVzLCBub2RlX2RpbV0g4oCUIHZpZW5lIGRlbCBDYXVzYWxSZWFzb25pbmdFbmdpbmUu'
        'CiAgICBlbmNvZGVyX2NvbmNlcHRzIFtCLCBMX2VuYywgaGlkZGVuX2RpbV0g4oCUIGNvbmNlcHQg'
        'dmVjdG9ycyBkZWwgU3RyZWFtRW5jb2Rlci4KICAgIEVuIGNhZGEgSHlicmlkRGVjb2RlckxheWVy'
        'LCBsb3MgdG9rZW5zIGhhY2VuIGNyb3NzLWF0dGVudGlvbiBhIGFtYm9zOgogICAgICAgIDEuIGdy'
        'YXBoX3JlcHIgICAgICAg4oaSIGFuY2xhIHNlbcOhbnRpY2EgY2F1c2FsIChlc3RydWN0dXJhKQog'
        'ICAgICAgIDIuIGVuY29kZXJfY29uY2VwdHMg4oaSIGFuY2xhIGlkZW50aWRhZCBsw6l4aWNhICh0'
        'b2tlbnMgZGVsIGlucHV0KQogICAgU2luIGVuY29kZXJfY29uY2VwdHMsIGVsIGRlY29kZXIgcHVl'
        'ZGUgZ2VuZXJhciAiaW5jZW5kaW8iIGN1YW5kbyBlbCBpbnB1dAogICAgZGljZSAiZmllYnJlIiBw'
        'b3JxdWUgbGEgaWRlbnRpZGFkIGzDqXhpY2Egc2UgcGllcmRlIGVuIGNyeXN0YWxsaXplciDihpIg'
        'Q1JFLgoKQU5DSE9SIEhFQUQ6CiAgICBQcmVkaWNlIGEgcXXDqSBub2RvIGRlbCBncmFmbyAicGVy'
        'dGVuZWNlIiBjYWRhIHRva2VuIGdlbmVyYWRvLgogICAgRHVyYW50ZSBlbCBlbnRyZW5hbWllbnRv'
        'IHNlIHVzYSBjb21vIHNlw7FhbCBhdXhpbGlhciBwYXJhIGZvcnphcgogICAgcXVlIGxvcyB0b2tl'
        'bnMgc2VhbiBmaWVsZXMgYWwgcmF6b25hbWllbnRvIGNhdXNhbC4KCk9VVFBVVCBNRVRBIEhFQUQ6'
        'CiAgICBQcmVkaWNlIG1ldGFkYXRvcyBkZWwgb3V0cHV0OgogICAgICAgIGNvbmZpZGVuY2U6ICAg'
        'ICAgICAgIMK/cXXDqSB0YW4gc2VndXJvIGVzdMOhIGVsIG1vZGVsbz8KICAgICAgICBuZWVkc19j'
        'bGFyaWZpY2F0aW9uOiDCv25lY2VzaXRhIHBlZGlyIG3DoXMgaW5mb3JtYWNpw7NuPwoKV0VJR0hU'
        'IFRZSU5HOgogICAgbG1faGVhZC53ZWlnaHQgPSB0b2tlbl9lbWJlZGRpbmcud2VpZ2h0IChwcsOh'
        'Y3RpY2EgZXN0w6FuZGFyIGVuIExNcykuCiAgICBSZWR1Y2UgcGFyw6FtZXRyb3MgeSBtZWpvcmEg'
        'Y29udmVyZ2VuY2lhLgoKUE9TSUNJT05FUzoKICAgIFRhYmxhIGRlIGVtYmVkZGluZ3MgYXByZW5k'
        'aWRhIGRlIHRhbWHDsW8gbWF4X3NlcV9sZW4uCiAgICBNw6FzIHNpbXBsZSBxdWUgUm9QRS9BTGlC'
        'aSwgc3VmaWNpZW50ZSBwYXJhIGVsIHNjb3BlIGRlIEFJT04tQy4KIiIiCgpmcm9tIF9fZnV0dXJl'
        'X18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgc3lzCmltcG9ydCBvcwpzeXMucGF0aC5pbnNl'
        'cnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXykpKQoKZnJvbSBk'
        'YXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0CgppbXBv'
        'cnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpmcm9tIHRvcmNoLnV0aWxzLmNoZWNrcG9p'
        'bnQgaW1wb3J0IGNoZWNrcG9pbnQgYXMgX2NrcHQKCmZyb20gLmNvbmZpZyBpbXBvcnQgU3RyZWFt'
        'RGVjb2RlckNvbmZpZwpmcm9tIC5oeWJyaWRfbGF5ZXIgaW1wb3J0IEh5YnJpZERlY29kZXJMYXll'
        'cgpmcm9tIC5tZXRhX2hlYWQgaW1wb3J0IE1ldGFPdXRwdXQsIE91dHB1dE1ldGFIZWFkCgoKIyDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIAKIyBPVVRQVVQgQ09OVEFJTkVSCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpAZGF0YWNsYXNzCmNsYXNzIERlY29kZXJPdXRw'
        'dXQ6CiAgICAiIiIKICAgIFJlc3VsdGFkbyBkZWwgU3RyZWFtRGVjb2Rlci4KCiAgICBsb2dpdHM6'
        'ICAgICAgICAgICAgICAgW0IsIEwsIHZvY2FiX3NpemVdICAgIOKAlCBkaXN0cmlidWNpw7NuIHNv'
        'YnJlIHZvY2FidWxhcmlvCiAgICBhbmNob3JfbG9naXRzOiAgICAgICAgW0IsIEwsIG1heF9ncmFw'
        'aF9ub2Rlc10g4oCUIHF1w6kgbm9kbyBkZWwgZ3JhZm8gcHJvZHVjZSBjYWRhIHRva2VuCiAgICBj'
        'b25maWRlbmNlOiAgICAgICAgICAgW0JdIOKIiCBbMCwxXSAgICAgICAgICDigJQgY29uZmlhbnph'
        'IGRlbCBtb2RlbG8KICAgIG5lZWRzX2NsYXJpZmljYXRpb246ICBbQl0g4oiIIFswLDFdICAgICAg'
        'ICAgIOKAlCBzb2xpY2l0YXIgbcOhcyBpbmZvIGFsIHVzdWFyaW8KICAgICIiIgogICAgbG9naXRz'
        'OiAgICAgICAgICAgICAgdG9yY2guVGVuc29yCiAgICBhbmNob3JfbG9naXRzOiAgICAgICB0b3Jj'
        'aC5UZW5zb3IKICAgIGNvbmZpZGVuY2U6ICAgICAgICAgIHRvcmNoLlRlbnNvcgogICAgbmVlZHNf'
        'Y2xhcmlmaWNhdGlvbjogdG9yY2guVGVuc29yCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBTVFJFQU0gREVDT0RFUgojIOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gAoKY2xhc3MgU3RyZWFtRGVjb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBEZWNvZGlmaWNh'
        'ZG9yIGF1dG9yZWdyZXNpdm8gY29uZGljaW9uYWRvIGVuIGVsIGdyYWZvIGNhdXNhbC4KCiAgICBV'
        'c286CiAgICAgICAgY29uZmlnICA9IFN0cmVhbURlY29kZXJDb25maWcoKQogICAgICAgIGRlY29k'
        'ZXIgPSBTdHJlYW1EZWNvZGVyKGNvbmZpZykKCiAgICAgICAgIyBncmFwaF9yZXByIGRlbCBDYXVz'
        'YWxSZWFzb25pbmdFbmdpbmUgKENSRU91dHB1dC5ub2RlX2ZlYXR1cmVzKQogICAgICAgIHRva2Vu'
        'X2lkcyAgPSB0b3JjaC5yYW5kaW50KDAsIGNvbmZpZy52b2NhYl9zaXplLCAoMiwgMTYpKQogICAg'
        'ICAgIGdyYXBoX3JlcHIgPSB0b3JjaC5yYW5kbigyLCA4LCBjb25maWcubm9kZV9kaW0pCgogICAg'
        'ICAgIG91dCA9IGRlY29kZXIodG9rZW5faWRzLCBncmFwaF9yZXByKQogICAgICAgICMgb3V0Lmxv'
        'Z2l0czogICAgICAgICBbMiwgMTYsIDMyMDAwXQogICAgICAgICMgb3V0LmFuY2hvcl9sb2dpdHM6'
        'ICBbMiwgMTYsIDMyXQogICAgICAgICMgb3V0LmNvbmZpZGVuY2U6ICAgICBbMl0KICAgICIiIgoK'
        'ICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWc6IFN0cmVhbURlY29kZXJDb25maWcpIC0+IE5v'
        'bmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25m'
        'aWcKICAgICAgICBEID0gY29uZmlnLmhpZGRlbl9kaW0KCiAgICAgICAgIyDilIDilIAgRW1iZWRk'
        'aW5ncyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIAKICAgICAgICBzZWxmLnRva2VuX2VtYmVkZGluZyA9IG5uLkVtYmVkZGluZyhjb25maWcudm9j'
        'YWJfc2l6ZSwgRCkKICAgICAgICBzZWxmLnBvc19lbWJlZGRpbmcgICA9IG5uLkVtYmVkZGluZyhj'
        'b25maWcubWF4X3NlcV9sZW4sIEQpCgogICAgICAgICMg4pSA4pSAIENhcGFzIGjDrWJyaWRhcyDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBzZWxmLmxh'
        'eWVyczogbm4uTW9kdWxlTGlzdCA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBIeWJyaWRE'
        'ZWNvZGVyTGF5ZXIoY29uZmlnKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShjb25maWcubl9s'
        'YXllcnMpCiAgICAgICAgXSkKCiAgICAgICAgIyDilIDilIAgTm9ybWFsaXphY2nDs24gZmluYWwg'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgc2VsZi5maW5hbF9ub3JtID0gbm4u'
        'TGF5ZXJOb3JtKEQsIGVwcz1jb25maWcubm9ybV9lcHMpCgogICAgICAgICMg4pSA4pSAIEhlYWRz'
        'IGRlIHNhbGlkYSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAg'
        'ICAjIExNIGhlYWQ6IHByZWRpY2UgZWwgc2lndWllbnRlIHRva2VuCiAgICAgICAgc2VsZi5sbV9o'
        'ZWFkID0gbm4uTGluZWFyKEQsIGNvbmZpZy52b2NhYl9zaXplLCBiaWFzPUZhbHNlKQoKICAgICAg'
        'ICAjIEFuY2hvciBoZWFkOiBxdcOpIG5vZG8gZGVsIGdyYWZvICJwcm9kdWNlIiBjYWRhIHRva2Vu'
        'CiAgICAgICAgc2VsZi5hbmNob3JfaGVhZCA9IG5uLkxpbmVhcihELCBjb25maWcubWF4X2dyYXBo'
        'X25vZGVzKQoKICAgICAgICAjIE1ldGEgaGVhZDogY29uZmlhbnphIHkgbmVjZXNpZGFkIGRlIGNs'
        'YXJpZmljYWNpw7NuCiAgICAgICAgc2VsZi5tZXRhX2hlYWQgPSBPdXRwdXRNZXRhSGVhZChjb25m'
        'aWcpCgogICAgICAgICMg4pSA4pSAIFdlaWdodCB0eWluZyDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICAjIHRva2VuX2VtYmVkZGluZy53ZWln'
        'aHQgW3ZvY2FiX3NpemUsIERdID09IGxtX2hlYWQud2VpZ2h0IFt2b2NhYl9zaXplLCBEXQogICAg'
        'ICAgICMgUHLDoWN0aWNhIGVzdMOhbmRhciBlbiBMTXM6IHJlZHVjZSBwYXLDoW1ldHJvcyB5IG1l'
        'am9yYSBjb252ZXJnZW5jaWEuCiAgICAgICAgc2VsZi5sbV9oZWFkLndlaWdodCA9IHNlbGYudG9r'
        'ZW5fZW1iZWRkaW5nLndlaWdodAoKICAgICAgICBzZWxmLmdyYWRpZW50X2NoZWNrcG9pbnRpbmcg'
        'PSBGYWxzZQogICAgICAgIHNlbGYuX2luaXRfd2VpZ2h0cygpCgogICAgZGVmIGVuYWJsZV9ncmFk'
        'aWVudF9jaGVja3BvaW50aW5nKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIiIiCiAgICAgICAgQWN0'
        'aXZhIGdyYWRpZW50IGNoZWNrcG9pbnRpbmcgZW4gbGFzIGNhcGFzIEh5YnJpZERlY29kZXJMYXll'
        'ci4KICAgICAgICBSZWNvbXB1dGEgbGFzIGFjdGl2YWNpb25lcyBkdXJhbnRlIGVsIGJhY2t3YXJk'
        'IGVuIGx1Z2FyIGRlIGd1YXJkYXJsYXMsCiAgICAgICAgcmVkdWNpZW5kbyBlbCB1c28gZGUgVlJB'
        'TSBkZWwgYmFja3dhcmQgfjItM8OXIGEgY29zdGEgZGUgfjE1JSBtw6FzIGNvbXB1dGUuCiAgICAg'
        'ICAgU29sbyBhY3Rpdm8gZW4gbW9kbyB0cmFpbmluZy4KICAgICAgICAiIiIKICAgICAgICBzZWxm'
        'LmdyYWRpZW50X2NoZWNrcG9pbnRpbmcgPSBUcnVlCgogICAgZGVmIF9pbml0X3dlaWdodHMoc2Vs'
        'ZikgLT4gTm9uZToKICAgICAgICAiIiJJbmljaWFsaXphY2nDs24gZXN0w6FuZGFyIHBhcmEgZW1i'
        'ZWRkaW5ncyB5IGhlYWRzLiIiIgogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLnRva2VuX2Vt'
        'YmVkZGluZy53ZWlnaHQsIHN0ZD0wLjAyKQogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLnBv'
        'c19lbWJlZGRpbmcud2VpZ2h0LCBzdGQ9MC4wMikKICAgICAgICAjIGxtX2hlYWQud2VpZ2h0IGVz'
        'IGVsIG1pc21vIHRlbnNvciBxdWUgdG9rZW5fZW1iZWRkaW5nLndlaWdodCAodHlpbmcpCiAgICAg'
        'ICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYuYW5jaG9yX2hlYWQud2VpZ2h0LCBzdGQ9MC4wMikKICAg'
        'ICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmFuY2hvcl9oZWFkLmJpYXMpCgogICAgZGVmIGZvcndh'
        'cmQoCiAgICAgICAgc2VsZiwKICAgICAgICB0b2tlbl9pZHM6ICAgICAgICB0b3JjaC5UZW5zb3Is'
        'ICAgICAgICAgICAgICAjIFtCLCBMXQogICAgICAgIGdyYXBoX3JlcHI6ICAgICAgIHRvcmNoLlRl'
        'bnNvciwgICAgICAgICAgICAgICMgW0IsIG5fbm9kZXMsIG5vZGVfZGltXQogICAgICAgIGVuY29k'
        'ZXJfY29uY2VwdHM6IHRvcmNoLlRlbnNvciB8IE5vbmUgPSBOb25lLCAgIyBbQiwgTF9lbmMsIGhp'
        'ZGRlbl9kaW1dCiAgICApIC0+IERlY29kZXJPdXRwdXQ6CiAgICAgICAgIiIiCiAgICAgICAgUGFz'
        'YSBwb3IgdG9kYXMgbGFzIGNhcGFzIHkgZ2VuZXJhIGxvZ2l0cyBwYXJhIGNhZGEgcG9zaWNpw7Nu'
        'LgoKICAgICAgICBBcmdzOgogICAgICAgICAgICB0b2tlbl9pZHM6ICAgICAgICBbQiwgTF0gICAg'
        'ICAgICAgICAgIOKAlCDDrW5kaWNlcyBkZSB0b2tlbnMgKHRlYWNoZXItZm9yY2VkKQogICAgICAg'
        'ICAgICBncmFwaF9yZXByOiAgICAgICBbQiwgbl9ub2RlcywgRF0gICAgIOKAlCByZXByZXNlbnRh'
        'Y2nDs24gZGVsIGdyYWZvIGNhdXNhbAogICAgICAgICAgICBlbmNvZGVyX2NvbmNlcHRzOiBbQiwg'
        'TF9lbmMsIERdICAgICAgIOKAlCBjb25jZXB0IHZlY3RvcnMgZGVsIFN0cmVhbUVuY29kZXIKICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgKG9wdGlvbmFsOyB3aGVuIE5vbmUsIGVuY29kZXIg'
        'Y3Jvc3MtYXR0ZW50aW9uIGlzIHNraXBwZWQpCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAg'
        'IERlY29kZXJPdXRwdXQgY29uIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0geSBtZXRhZGF0b3MK'
        'ICAgICAgICAiIiIKICAgICAgICBCLCBMID0gdG9rZW5faWRzLnNoYXBlCgogICAgICAgICMgUG9z'
        'aXRpb25zOiBbMSwgTF0g4oCUIGJyb2FkY2FzdCBzb2JyZSBlbCBiYXRjaAogICAgICAgIHBvcyA9'
        'IHRvcmNoLmFyYW5nZShMLCBkZXZpY2U9dG9rZW5faWRzLmRldmljZSwgZHR5cGU9dG9yY2gubG9u'
        'ZykudW5zcXVlZXplKDApCgogICAgICAgICMgQ29tYmluYXIgdG9rZW4gZW1iZWRkaW5nICsgcG9z'
        'aXRpb25hbCBlbWJlZGRpbmcKICAgICAgICB4ID0gc2VsZi50b2tlbl9lbWJlZGRpbmcodG9rZW5f'
        'aWRzKSArIHNlbGYucG9zX2VtYmVkZGluZyhwb3MpICAjIFtCLCBMLCBEXQoKICAgICAgICAjIFBh'
        'c2FyIHBvciB0b2RhcyBsYXMgSHlicmlkRGVjb2RlckxheWVycwogICAgICAgIGZvciBsYXllciBp'
        'biBzZWxmLmxheWVyczoKICAgICAgICAgICAgaWYgc2VsZi5ncmFkaWVudF9jaGVja3BvaW50aW5n'
        'IGFuZCBzZWxmLnRyYWluaW5nOgogICAgICAgICAgICAgICAgeCA9IF9ja3B0KGxheWVyLCB4LCBn'
        'cmFwaF9yZXByLCBlbmNvZGVyX2NvbmNlcHRzLAogICAgICAgICAgICAgICAgICAgICAgICAgIHVz'
        'ZV9yZWVudHJhbnQ9RmFsc2UpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB4ID0g'
        'bGF5ZXIoeCwgZ3JhcGhfcmVwciwgZW5jb2Rlcl9jb25jZXB0cykgICAjIFtCLCBMLCBEXQoKICAg'
        'ICAgICAjIE5vcm1hbGl6YWNpw7NuIGZpbmFsCiAgICAgICAgeCA9IHNlbGYuZmluYWxfbm9ybSh4'
        'KSAgICAgICAgICMgW0IsIEwsIERdCgogICAgICAgICMgSGVhZHMgZGUgc2FsaWRhCiAgICAgICAg'
        'bG9naXRzICAgICAgICA9IHNlbGYubG1faGVhZCh4KSAgICAgICAgIyBbQiwgTCwgdm9jYWJfc2l6'
        'ZV0KICAgICAgICBhbmNob3JfbG9naXRzID0gc2VsZi5hbmNob3JfaGVhZCh4KSAgICAjIFtCLCBM'
        'LCBtYXhfZ3JhcGhfbm9kZXNdCiAgICAgICAgbWV0YSAgICAgICAgICA9IHNlbGYubWV0YV9oZWFk'
        'KHgsIGdyYXBoX3JlcHIpCgogICAgICAgIHJldHVybiBEZWNvZGVyT3V0cHV0KAogICAgICAgICAg'
        'ICBsb2dpdHMgICAgICAgICAgICAgID0gbG9naXRzLAogICAgICAgICAgICBhbmNob3JfbG9naXRz'
        'ICAgICAgID0gYW5jaG9yX2xvZ2l0cywKICAgICAgICAgICAgY29uZmlkZW5jZSAgICAgICAgICA9'
        'IG1ldGEuY29uZmlkZW5jZSwKICAgICAgICAgICAgbmVlZHNfY2xhcmlmaWNhdGlvbiA9IG1ldGEu'
        'bmVlZHNfY2xhcmlmaWNhdGlvbiwKICAgICAgICApCgogICAgIyDilIDilIAgVXRpbGlkYWRlcyDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIAKCiAgICBkZWYgY291bnRfcGFyYW1ldGVycyhzZWxmKSAtPiBpbnQ6CiAgICAgICAg'
        'IiIiTsO6bWVybyB0b3RhbCBkZSBwYXLDoW1ldHJvcyBlbnRyZW5hYmxlcy4iIiIKICAgICAgICAj'
        'IHRva2VuX2VtYmVkZGluZyB5IGxtX2hlYWQgY29tcGFydGVuIHBlc29zIOKGkiBjb250YXIgdW5h'
        'IHNvbGEgdmV6CiAgICAgICAgc2Vlbjogc2V0ID0gc2V0KCkKICAgICAgICB0b3RhbCA9IDAKICAg'
        'ICAgICBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgaWYgaWQocCkgbm90'
        'IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZChpZChwKSkKICAgICAgICAgICAgICAg'
        'IGlmIHAucmVxdWlyZXNfZ3JhZDoKICAgICAgICAgICAgICAgICAgICB0b3RhbCArPSBwLm51bWVs'
        'KCkKICAgICAgICByZXR1cm4gdG90YWwKCiAgICBkZWYgcGFyYW1ldGVyX2JyZWFrZG93bihzZWxm'
        'KSAtPiBkaWN0OgogICAgICAgICIiIkRlc2dsb3NlIGRlIHBhcsOhbWV0cm9zIHBvciBzdWItbcOz'
        'ZHVsbyAoc2luIGRvYmxlLWNvbnRhciB3ZWlnaHQgdHlpbmcpLiIiIgogICAgICAgIGVtYmVkID0g'
        'c2VsZi50b2tlbl9lbWJlZGRpbmcud2VpZ2h0Lm51bWVsKCkKICAgICAgICBwb3MgICA9IHNlbGYu'
        'cG9zX2VtYmVkZGluZy53ZWlnaHQubnVtZWwoKQogICAgICAgIGxheWVyc190b3RhbCA9IHN1bSgK'
        'ICAgICAgICAgICAgcC5udW1lbCgpIGZvciBsYXllciBpbiBzZWxmLmxheWVycyBmb3IgcCBpbiBs'
        'YXllci5wYXJhbWV0ZXJzKCkKICAgICAgICApCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAg'
        'InRva2VuX2VtYmVkZGluZyI6ICAgZW1iZWQsCiAgICAgICAgICAgICJwb3NfZW1iZWRkaW5nIjog'
        'ICAgIHBvcywKICAgICAgICAgICAgImxheWVycyI6ICAgICAgICAgICAgbGF5ZXJzX3RvdGFsLAog'
        'ICAgICAgICAgICAibG1faGVhZCI6ICAgICAgICAgICAwLCAgIyB0aWVkIGNvbiB0b2tlbl9lbWJl'
        'ZGRpbmcKICAgICAgICAgICAgImFuY2hvcl9oZWFkIjogICAgICAgc3VtKHAubnVtZWwoKSBmb3Ig'
        'cCBpbiBzZWxmLmFuY2hvcl9oZWFkLnBhcmFtZXRlcnMoKSksCiAgICAgICAgICAgICJtZXRhX2hl'
        'YWQiOiAgICAgICAgIHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5tZXRhX2hlYWQucGFyYW1l'
        'dGVycygpKSwKICAgICAgICAgICAgInRvdGFsX3VuaXF1ZSI6ICAgICAgc2VsZi5jb3VudF9wYXJh'
        'bWV0ZXJzKCksCiAgICAgICAgfQo='
    ),
    'router/__init__.py': (
        'IiIiCkFJT04tQyByb3V0ZXIg4oCUIENPUkFQaXBlbGluZSBlbmQtdG8tZW5kLgoKQ29uZWN0YSBT'
        'RSDihpIgR0Mg4oaSIENSRSDihpIgU0QgZW4gdW4gbcOzZHVsbyDDum5pY28gZGlmZXJlbmNpYWJs'
        'ZS4KIiIiCgpmcm9tIC5waXBlbGluZSBpbXBvcnQgQ09SQUNvbmZpZywgQ09SQVBpcGVsaW5lLCBQ'
        'aXBlbGluZU91dHB1dAoKX19hbGxfXyA9IFsKICAgICJDT1JBQ29uZmlnIiwKICAgICJDT1JBUGlw'
        'ZWxpbmUiLAogICAgIlBpcGVsaW5lT3V0cHV0IiwKXQo='
    ),
    'router/pipeline.py': (
        '...'
        'c3MK'
    ),
    'experiments/__init__.py': (
        'IyBleHBlcmltZW50cyBwYWNrYWdlCg=='
    ),
    'experiments/train_cora_50m.py': (
        '...'
    ),
}

for rel_path, b64_content in files_b64.items():
    if isinstance(b64_content, tuple):
        b64_content = ''.join(b64_content)
    content = base64.b64decode(b64_content).decode('utf-8')
    (ROOT / rel_path).write_text(content, encoding='utf-8')
    print(f'  ✓ {rel_path}')

sys.path.insert(0, str(ROOT))
print(f'\n✓ {len(files_b64)} archivos escritos en {ROOT}')
print('✓ sys.path configurado')


In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM: {free/1e9:.1f}GB libre / {total/1e9:.1f}GB total')
    # Estimate model VRAM
    N_PARAMS = 37_053_274
    model_mb = N_PARAMS * 4 / 1e6
    optim_mb = N_PARAMS * 4 * 2 / 1e6
    activ_mb = 400
    total_est = model_mb + optim_mb + activ_mb
    print(f'\nEstimacion VRAM del modelo:')
    print(f'  Modelo (f32):      {model_mb:.0f} MB')
    print(f'  Optimizer (AdamW): {optim_mb:.0f} MB')
    print(f'  Activaciones:      {activ_mb} MB')
    print(f'  TOTAL estimado:    {total_est:.0f} MB ({total_est/1024:.2f} GB)')
    if total_est/1024 < free/1e9:
        print(f'  \u2713 Cabe en VRAM disponible')
    else:
        print(f'  \u26a0 ADVERTENCIA: puede no caber')
else:
    print('\u26a0 Sin GPU. Ve a Runtime \u2192 Change runtime type \u2192 T4 GPU')

In [ ]:
import sys
sys.path.insert(0, '/content/aion_c')

# Import and run training
import experiments.train_cora_50m as trainer

# Override constants for T4
trainer.N_DATASET    = 5000
trainer.N_STEPS      = 5000
trainer.ACCUM_STEPS  = 4
trainer.EVAL_EVERY   = 500
trainer.CKPT_EVERY   = 1000
trainer.PRINT_EVERY  = 100
trainer.CRE_ITERS    = 10
trainer.TRAIN_FRAC   = 0.80

# Override checkpoint dir to use Google Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/cora_50m'
original_save = trainer.save_checkpoint

def drive_checkpoint(*args, **kwargs):
    kwargs['ckpt_dir'] = f'{DRIVE_DIR}/checkpoints'
    return original_save(*args, **kwargs)

trainer.save_checkpoint = drive_checkpoint

# Run training
trainer.train()

In [ ]:
import json, glob, matplotlib
import matplotlib.pyplot as plt

# Find latest results file
results_files = sorted(glob.glob('/content/aion_c/experiments/results/train_cora_50m_*.json'))
if not results_files:
    results_files = sorted(glob.glob(f'{DRIVE_DIR}/results/train_cora_50m_*.json'))

if results_files:
    with open(results_files[-1]) as f:
        data = json.load(f)

    # Plot loss curve
    steps = [r['step'] for r in data['loss_curve']]
    total_loss = [r['total'] for r in data['loss_curve']]
    lm_loss = [r['lm'] for r in data['loss_curve'] if r.get('lm')]
    lm_steps = [r['step'] for r in data['loss_curve'] if r.get('lm')]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(steps, total_loss, alpha=0.4, color='gray', linewidth=0.8, label='raw')
    # Smooth with rolling average
    window = 50
    smoothed = [sum(total_loss[max(0,i-window):i+1])/len(total_loss[max(0,i-window):i+1])
                for i in range(len(total_loss))]
    axes[0].plot(steps, smoothed, color='steelblue', linewidth=2, label=f'MA-{window}')
    axes[0].set_title('Loss Total')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if lm_loss:
        lm_smooth = [sum(lm_loss[max(0,i-window):i+1])/len(lm_loss[max(0,i-window):i+1])
                     for i in range(len(lm_loss))]
        axes[1].plot(lm_steps, lm_loss, alpha=0.4, color='gray', linewidth=0.8)
        axes[1].plot(lm_steps, lm_smooth, color='coral', linewidth=2)
        axes[1].set_title('LM Loss (generación de respuestas)')
        axes[1].set_xlabel('Step')
        axes[1].grid(alpha=0.3)

    plt.suptitle('CORA 50M — Curva de entrenamiento', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Show eval snapshots
    print('\n' + '='*70)
    print('EVALUACIONES DURANTE ENTRENAMIENTO')
    print('='*70)

    for snap in data.get('eval_snapshots', []):
        print(f"\n--- Step {snap['step']} ---")
        print(f"node_acc={snap['avg_node_acc']:.1%}  rel_recall={snap['avg_rel_recall']:.1%}  word_F1={snap['avg_word_f1']:.1%}")

    # Show last 5 eval examples from last snapshot
    snaps = data.get('eval_snapshots', [])
    if snaps:
        last = snaps[-1]
        print(f'\n{"="*70}')
        print(f'ULTIMOS 5 EJEMPLOS EVAL (step {last["step"]})')
        print(f'{"="*70}')
        for i, ex in enumerate(last.get('examples', [])[:5], 1):
            print(f'\n[{i}] Nivel {ex["level"]}')
            print(f'  Input   : {ex["q_preview"]}...')
            print(f'  GT nodos: {ex["gt_n"]}  Pred nodos: {ex["pred_n"]}  NodeAcc: {ex["node_acc"]:.0%}')
            print(f'  GT      : {ex["gt_answer"][:80]}')
            print(f'  Generado: {ex["gen_answer"][:80] or "(vacío)"}')
            print(f'  Word F1 : {ex["word_overlap"]:.0%}')

else:
    print('No se encontraron archivos de resultados.')
    print('Asegúrate de haber ejecutado la celda de entrenamiento.')